In [ ]:
import sys
import os
import numpy as np
# Add the src directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
from src.pomdp.pomdp import POMDPFactory
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils import analyze_meg_draws, calc_num_draws,plot_num_draws,plot_best_actions_with_subject_choices


# To DO:
- [ ]  Look at the quantiles of confidence and the quantiles of correctness.
- [ ]  Scatter plot between my confidence value and their confidence values at the beggining before any transport.
- [ ]  Plot the subjects decisions, the difference in evidence over draw per subjects. and across subjects.
- [ ]  Also, their reward and confidence alteration. with the number of draws.
- [ ]  Ask all of the interesting questions regarding the human behavior, and accordingly explore the data. This step is essential for how to actually make our model replicates the human behavior.
- [ ]  Plot the propotion of people who decided at different location at the heat map I have.

## Some Statistics about the data

In [ ]:
IS_LATEX=False
def _set_plot_style(font_size=20, is_latex=IS_LATEX):
    """Helper to set consistent plot styles."""
    if is_latex:
        plt.rcParams.update(
            {
                "text.usetex": True,
                "font.family": "serif",
                "font.size": font_size,
                "axes.titlesize": font_size,
                "axes.labelsize": font_size,
                "xtick.labelsize": font_size,
                "ytick.labelsize": font_size,
                "legend.fontsize": font_size,
            }
        )
    else:
        # Reset to default or specify non-LaTeX styles here if needed
        plt.rcParams.update(
            {
                "font.size": font_size,
                "axes.titlesize": font_size,
                "axes.labelsize": font_size,
                "xtick.labelsize": font_size,
                "ytick.labelsize": font_size,
                "legend.fontsize": font_size,
            }
        )

In [ ]:
FIGURE_PATH = "../figures/"
# === Function 2 ===

def save_figure(fname, path=FIGURE_PATH, save_fig=True):
    """
    Helper function to save the current matplotlib figure.
    """
    if save_fig:
        os.makedirs(path, exist_ok=True)
        full_fname = os.path.join(path, fname)
        plt.savefig(full_fname, dpi=300, bbox_inches='tight')
        print(f"Figure saved to {full_fname}")


def plot_human_vs_simulated_data(outcome_human, num_draws_human,
                                 outcome_simulated, num_draws_simulated,
                                 bins_outcome=25,
                                 font_size=20,
                                 path=FIGURE_PATH, save_fig=True,
                                 fname="human_vs_simulated_overlay.pdf"):

    if is_latex:
        plt.rcParams.update({
            "text.usetex": True,
            "font.family": "serif",
            "font.size": font_size,
            "axes.titlesize": font_size,
            "axes.labelsize": font_size,
            "xtick.labelsize": font_size,
            "ytick.labelsize": font_size,
            "legend.fontsize": font_size,
        })

    fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=False)
    num_bins = np.arange(0, max(max(num_draws_human), max(num_draws_simulated)) + 2, 1)

    # Outcomes
    sns.histplot(outcome_human, bins=bins_outcome, color='skyblue',
                 label='Human', ax=axes[0], kde=True, stat="count", alpha=0.5)
    sns.histplot(outcome_simulated, bins=bins_outcome, color='salmon',
                 label='Simulated', ax=axes[0], kde=True, stat="count", alpha=0.5)
    axes[0].set_title('Outcome Values: Human vs. Simulated')
    axes[0].set_xlabel('Outcome')
    axes[0].set_ylabel('Density')
    axes[0].legend()

    # Draws
    sns.histplot(num_draws_human, bins=num_bins, color='skyblue',
                 label='Human', ax=axes[1], kde=False, stat="count", alpha=0.5)
    sns.histplot(num_draws_simulated, bins=num_bins, color='salmon',
                 label='Simulated', ax=axes[1], kde=False, stat="count", alpha=0.5)
    axes[1].set_title('Number of Draws: Human vs. Simulated')
    axes[1].set_xlabel('Number of Draws')
    axes[1].set_ylabel('Frequency')
    axes[1].set_xticks(num_bins)
    axes[1].legend()

    plt.tight_layout()
    plt.suptitle('Comparison: Human vs. Simulated Data', fontsize=font_size, y=1.02)

    save_figure(fname, path, save_fig)
    plt.show()

In [ ]:
def run_plot_action_values(counts_dict_short,horizon_condition,params={'tau':0.01,"xi":0.0},project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
):
    # Arguments:
    # horizon_condition='short'
    max_cards_per_draw=5
    verbose=False
    data_path = os.path.join(project_root, f"data/POMDP/illustrate")
    figure_path = os.path.join(project_root,f"figures/POMDP/illustrate")
    os.makedirs(data_path, exist_ok=True)
    os.makedirs(figure_path, exist_ok=True)
    is_hazardous=True
    params.update( {
        "is_hazardous": is_hazardous,
        "horizon_condition": horizon_condition,
        "verbose": verbose,
        "max_cards_per_draw": max_cards_per_draw,
    })
    pomdp=POMDPFactory("vanilla")
    pomdp.__init__(**params)
    pomdp.value_iteration()
    best_actions = pomdp.best_actions
    beliefs = pomdp.belief
    actionva=plot_best_actions_with_subject_choices(
        best_actions, max_cards_per_draw=max_cards_per_draw, path=figure_path,counts_dict=counts_dict_short,horizon_condition=horizon_condition
    )

# MEG Task

## Short horizon

In [ ]:
import pandas as pd
file_path = os.path.join(project_root, "data/TrHu_NHB_light/data_MEG/behdat_preprocessed.pkl")
human_meg_data = pd.read_pickle(file_path)
# access data using the userID
def get_subject_data(human_meg_data,user_id):
    subject_data = human_meg_data[human_meg_data['userID'] == user_id]
    if not subject_data.empty:
        return subject_data['data'].values[0]
    else:
        return None
    
# extract all the userID in the human_meg_data
userID_list= human_meg_data['userID'].unique().tolist()
print(f"Number of unique user IDs: {len(userID_list)}")


In [ ]:
subjec=get_subject_data(human_meg_data,28)
subjec.columns

In [ ]:
# get the block 1 game 13 for subjec
print(subjec[(subjec['block'] == 1) & (subjec['game'] == 13)])

In [ ]:
subj_data=get_subject_data(human_meg_data,28)
subj_data.head(30)


In [ ]:
counts=analyze_meg_draws(human_meg_data)

In [ ]:
pair_count_short=counts['pair_count_short']
pair_count_long=counts['pair_count_long']
counts_dict_short=counts['counts_dict_short']
counts_dict_long=counts['counts_dict_long']
num_games=counts['num_games']

In [ ]:

# the total number of games are: 
# sum up all occurances for all 
total_counts_short = sum(pair_count_short.values())
print(f"Total counts of last tuples: {total_counts_short}")
# sum up all occurances for all 
total_counts_long = sum(pair_count_long.values())
print(f"Total counts of last tuples: {total_counts_long}")
print(f"Total number of games: {num_games}")
print("sum of the short and long sequences:", total_counts_short + total_counts_long)
results=counts['results']

In [ ]:
import pandas as pd

stats = {'short': {'correct': 0, 'incorrect': 0, 'missed': 0},
         'long':  {'correct': 0, 'incorrect': 0, 'missed': 0}}

for user_id in userID_list:
    subj = get_subject_data(human_meg_data, user_id)
    if subj is None:
        continue
    for (block, game), game_data in subj.groupby(['block', 'game']):
        horizon = 'short' if len(game_data) <= 8 else 'long'

        reward_series = game_data['reward'].dropna()
        if reward_series.empty:
            # no reward recorded — treat as missed
            stats[horizon]['missed'] += 1
            continue
        r = int(reward_series.iloc[-1])
        if r == 2:
            stats[horizon]['correct'] += 1
        elif r == -2:
            stats[horizon]['incorrect'] += 1
        else:                       # r == -1
            stats[horizon]['missed'] += 1

rows = []
for cond in ('short', 'long'):
    s = stats[cond]
    total   = s['correct'] + s['incorrect'] + s['missed']
    acc     = s['correct'] / total if total > 0 else float('nan')
    miss_r  = s['missed']  / total if total > 0 else float('nan')
    rows.append({
        'Condition':          cond.capitalize(),
        'Correct':            s['correct'],
        'Incorrect (lapses)': s['incorrect'],
        'Missed':             s['missed'],
        'Total':              total,
        'Accuracy':           f"{acc:.3f}",
        'Missing Rate':       f"{miss_r:.3f}",
    })

decision_stats_df = pd.DataFrame(rows).set_index('Condition')
display(decision_stats_df)


In [ ]:
# Auto-generate LaTeX table and paragraph from decision_stats_df

# ── LaTeX table ──────────────────────────────────────────────────────────────
latex_str = decision_stats_df.to_latex(
    caption="Decision statistics broken down by horizon condition. "
            "Accuracy $= \\text{Correct}/\\text{Total}$; "
            "Missing Rate $= \\text{Missed}/\\text{Total}$.",
    label="tab:decision_stats",
    column_format="lrrrrrr",
    bold_rows=True,
)
print(latex_str)

# ── Pull values for the paragraph ────────────────────────────────────────────
s = decision_stats_df.loc['Short']
l = decision_stats_df.loc['Long']

acc_s      = float(s['Accuracy'])
acc_l      = float(l['Accuracy'])
miss_s     = float(s['Missing Rate'])
miss_l     = float(l['Missing Rate'])
inc_s      = int(s['Incorrect (lapses)'])
inc_l      = int(l['Incorrect (lapses)'])
total      = int(s['Total'])
cor_s      = int(s['Correct'])
cor_l      = int(l['Correct'])

# ── Auto-generated paragraph ──────────────────────────────────────────────────
paragraph = (
    f"Participants completed {total:,} trials in each horizon condition. "
    f"Overall task performance was high, with accuracy (correct decisions / total trials) "
    f"of {acc_s:.1%} in the short-horizon condition and {acc_l:.1%} in the long-horizon condition "
    f"(Table~\\ref{{tab:decision_stats}}). "
    f"The missing-response rate --- trials in which no decision was recorded before the deadline --- "
    f"differed markedly between conditions ({miss_s:.1%} short vs.\\ {miss_l:.1%} long), "
    f"suggesting that participants were more likely to withhold decisions under time pressure "
    f"in the short-horizon condition. "
    f"The number of incorrect responses (lapses) was similar across conditions "
    f"(${inc_s:,}$ short vs.\\ ${inc_l:,}$ long), "
    f"consistent with lapses reflecting a condition-independent noise process rather than "
    f"a strategic response to horizon length. "
    f"The substantially higher accuracy and lower missing rate in the long-horizon condition "
    f"are consistent with participants having more evidence available before committing to a decision, "
    f"in line with the POMDP framework's prediction that extended observation opportunities "
    f"improve belief precision."
)

print("\n% ── Paragraph ────────────────────────────────────────────────────────────")
print(paragraph)


## The fraction of the people who passed through the num yellow and blue and didn't decide

In [ ]:

run_plot_action_values(counts_dict_short,horizon_condition='short')

In [ ]:

run_plot_action_values(counts_dict_long,horizon_condition='long')

In [ ]:
from src.utils import analyze_meg_draws_color

color_counts = analyze_meg_draws_color(human_meg_data)

counts_dict_yellow_short = color_counts['counts_dict_yellow_short']
counts_dict_yellow_long  = color_counts['counts_dict_yellow_long']
counts_dict_blue_short   = color_counts['counts_dict_blue_short']
counts_dict_blue_long    = color_counts['counts_dict_blue_long']


In [ ]:

# def plot_yellow_blue_fractions_stacked(
#     best_actions_short,
#     best_actions_long,
#     max_cards_per_draw,
#     counts_dict_yellow_short,
#     counts_dict_blue_short,
#     counts_dict_yellow_long,
#     counts_dict_blue_long,
#     label=None,
#     path="../figures/illustrate",
#     min_visits=0,
#     ytick_step_short=5,
#     ytick_step_long=10,
#     ylim_short=25,
#     ylim_long=40,
#     bg_color="#f0f0f0",
# ):
#     """
#     Six-panel heatmap: short horizon (top row) and long horizon (bottom row),
#     each row showing fraction-wait / fraction-yellow / fraction-blue.

#     The x-axis (number of draws) is shared across rows; the short horizon is
#     padded with masked columns to match the long-horizon draw range so the
#     shared axis is meaningful.

#     Figure background is light grey to visually separate masked (white) cells
#     from the figure margins.
#     """
#     from matplotlib.colors import LinearSegmentedColormap

#     _set_plot_style()

#     BG_COLOR = bg_color

#     cmap_wait   = LinearSegmentedColormap.from_list("wait",   ["white", "#1a8c1a"])
#     cmap_yellow = LinearSegmentedColormap.from_list("yellow", ["white", "#FFD700"])
#     cmap_blue   = LinearSegmentedColormap.from_list("blue",   ["white", "#0033cc"])
#     cmaps = [cmap_wait, cmap_yellow, cmap_blue]
#     titles = ["Fraction Wait", "Fraction Yellow", "Fraction Blue"]
#     cbar_labels = ["Wait", "Fraction", "Fraction"]

#     def _build_arrays(best_actions, counts_dict_yellow, counts_dict_blue,
#                       n_draws_out, ytick_step):
#         num_draws, num_yellow, num_blue = best_actions.shape
#         max_diff   = num_yellow - 1
#         min_diff   = -max_diff
#         diff_range = np.arange(min_diff, max_diff + 1)
#         n_diff     = len(diff_range)

#         f_wait   = np.full((n_draws_out, n_diff), np.nan)
#         f_yellow = np.full((n_draws_out, n_diff), np.nan)
#         f_blue   = np.full((n_draws_out, n_diff), np.nan)
#         mask     = np.ones((n_draws_out, n_diff), dtype=bool)

#         for draw in range(1, num_draws):
#             if draw - 1 >= n_draws_out:
#                 break
#             for yellow in range(num_yellow):
#                 blue = draw * max_cards_per_draw - yellow
#                 if blue < 0 or blue >= num_blue:
#                     continue
#                 diff = yellow - blue
#                 if not (min_diff <= diff <= max_diff):
#                     continue
#                 key = (float(yellow), float(blue))
#                 if key not in counts_dict_yellow or key not in counts_dict_blue:
#                     continue
#                 n_y, denom = counts_dict_yellow[key].split("/")
#                 n_b        = counts_dict_blue[key].split("/")[0]
#                 denom, n_y, n_b = int(denom), int(n_y), int(n_b)
#                 if denom < min_visits:
#                     continue
#                 col_idx = diff - min_diff
#                 f_y = n_y / denom
#                 f_b = n_b / denom
#                 f_yellow[draw - 1, col_idx] = f_y
#                 f_blue  [draw - 1, col_idx] = f_b
#                 f_wait  [draw - 1, col_idx] = max(0.0, 1.0 - f_y - f_b)
#                 mask    [draw - 1, col_idx] = False

#         indices = [i for i, v in enumerate(diff_range) if v % ytick_step == 0]
#         return (
#             [np.ma.array(f_wait,   mask=mask),
#              np.ma.array(f_yellow, mask=mask),
#              np.ma.array(f_blue,   mask=mask)],
#             diff_range, indices, n_diff,
#         )

#     # Use long-horizon draw count as the shared x dimension
#     n_draws_long = best_actions_long.shape[0] - 1
#     n_draws_short_raw = best_actions_short.shape[0] - 1

#     arrays_s, diff_range_s, idx_s, n_diff_s = _build_arrays(
#         best_actions_short,
#         counts_dict_yellow_short, counts_dict_blue_short,
#         n_draws_long, ytick_step_short,
#     )
#     arrays_l, diff_range_l, idx_l, n_diff_l = _build_arrays(
#         best_actions_long,
#         counts_dict_yellow_long, counts_dict_blue_long,
#         n_draws_long, ytick_step_long,
#     )

#     fig, axes = plt.subplots(
#         2, 3,
#         figsize=(30, 14),
#         sharex=True,
#     )
#     fig.patch.set_facecolor('white')

#     xtick_pos    = np.arange(0, n_draws_long)
#     xtick_labels = np.arange(1, n_draws_long + 1)

#     horizon_configs = [
#         (arrays_s, diff_range_s, idx_s, n_diff_s, ytick_step_short, "Short", ylim_short),
#         (arrays_l, diff_range_l, idx_l, n_diff_l, ytick_step_long,  "Long",  ylim_long),
#     ]

#     for col, (cmap_p, title, cbar_lbl) in enumerate(zip(cmaps, titles, cbar_labels)):
#         for row, (arrays, diff_range, indices, n_diff, ytick_step, horizon, ylim) in enumerate(horizon_configs):
#             ax = axes[row][col]
#             ax.set_facecolor(BG_COLOR)

#             im = ax.imshow(
#                 arrays[col].T,
#                 cmap=cmap_p,
#                 aspect="auto",
#                 interpolation="nearest",
#                 origin="lower",
#                 vmin=0, vmax=1,
#             )
#             ax.set_ylabel("Yellow − Blue")
#             ax.set_title(f"{title} ({horizon} Horizon)")

#             ax.set_yticks(indices)
#             ax.set_yticklabels(diff_range[indices])

#             ax.grid(False)
#             ax.set_xticks(np.arange(-0.5, n_draws_long), minor=True)
#             ax.set_yticks(np.arange(-0.5, n_diff), minor=True)
#             ax.tick_params(which="minor", bottom=False, left=False)

#             cbar = fig.colorbar(im, ax=ax, ticks=[0, 0.5, 1])
#             cbar.set_label(cbar_lbl)

#             # Horizontal line at Yellow − Blue = 0
#             zero_idx = list(diff_range).index(0)
#             ax.axhline(y=zero_idx, color='black', linestyle='-',
#                        linewidth=1.2, alpha=0.8)

#             # Y-axis limits in data units (Yellow − Blue difference)
#             y_lo = zero_idx - ylim
#             y_hi = zero_idx + ylim
#             ax.set_ylim(max(y_lo, -0.5), min(y_hi, n_diff - 0.5))

#             # Only bottom row gets x-tick labels (shared axis)
#             if row == 1:
#                 ax.set_xticks(xtick_pos)
#                 ax.set_xticklabels(xtick_labels)
#                 ax.set_xlabel("Number of Draws")
#             else:
#                 ax.set_xticks(xtick_pos)
#                 ax.set_xticklabels([])
#                 # Shade columns beyond the short-horizon deadline (draw > 8)
#                 ax.axvspan(n_draws_short_raw - 0.5, n_draws_long - 0.5,
#                            color='grey', alpha=0.45, zorder=2)
#                 ax.axvline(x=n_draws_short_raw - 0.5, color='black',
#                            linestyle='--', linewidth=1.5, alpha=0.9, zorder=3)

#     fig.tight_layout()

#     base = f"{path}/yellow_blue_fractions_stacked"
#     if label is not None:
#         base += f"_{label}"
#     fig.savefig(base + ".pdf", bbox_inches="tight", pad_inches=0.03)
#     fig.savefig(base + ".svg", bbox_inches="tight", pad_inches=0.03)
#     fig.savefig(base + ".png", dpi=600, bbox_inches="tight", pad_inches=0.03)
#     plt.show()


In [ ]:
from src.utils.plotting import plot_yellow_blue_fractions_stacked

def run_plot_yellow_blue_stacked(
    counts_dict_yellow_short, counts_dict_blue_short,
    counts_dict_yellow_long,  counts_dict_blue_long,
    params=None,
    ytick_step_short=5,
    ytick_step_long=10,
    project_root=os.path.abspath(os.path.join(os.getcwd(), '..')),
    dark=False,
):
    if params is None:
        params = {'tau': 0.01, 'xi': 0.0}
    max_cards_per_draw = 5
    figure_path = os.path.join(project_root, 'figures/POMDP/illustrate')
    os.makedirs(figure_path, exist_ok=True)

    def _make_pomdp(horizon_condition):
        p = dict(params)
        p.update({
            'is_hazardous':       True,
            'horizon_condition':  horizon_condition,
            'verbose':            False,
            'max_cards_per_draw': max_cards_per_draw,
        })
        pomdp = POMDPFactory('vanilla')
        pomdp.__init__(**p)
        pomdp.value_iteration()
        return pomdp

    pomdp_short = _make_pomdp('short')
    pomdp_long  = _make_pomdp('long')

    plot_yellow_blue_fractions_stacked(
        best_actions_short=pomdp_short.best_actions,
        best_actions_long=pomdp_long.best_actions,
        max_cards_per_draw=max_cards_per_draw,
        counts_dict_yellow_short=counts_dict_yellow_short,
        counts_dict_blue_short=counts_dict_blue_short,
        counts_dict_yellow_long=counts_dict_yellow_long,
        counts_dict_blue_long=counts_dict_blue_long,
        path=figure_path,
        ytick_step_short=ytick_step_short,
        ytick_step_long=ytick_step_long,
        ylim_long=30, ylim_short=30,
        dark=dark,bg_color="lightgray"
    )


run_plot_yellow_blue_stacked(
    counts_dict_yellow_short, counts_dict_blue_short,
    counts_dict_yellow_long,  counts_dict_blue_long,
    dark=False,
)


In [ ]:
from src.utils.plotting import plot_yellow_blue_fractions_separate

def run_plot_yellow_blue_separate(
    counts_dict_yellow_short, counts_dict_blue_short,
    counts_dict_yellow_long,  counts_dict_blue_long,
    params=None,
    ytick_step_short=5,
    ytick_step_long=10,
    project_root=os.path.abspath(os.path.join(os.getcwd(), '..')),
    dark=True,
    font_size=26,
):
    if params is None:
        params = {'tau': 0.01, 'xi': 0.0}
    max_cards_per_draw = 5
    figure_path = os.path.join(project_root, 'figures/POMDP/illustrate')
    os.makedirs(figure_path, exist_ok=True)

    def _make_pomdp(horizon_condition):
        p = dict(params)
        p.update({
            'is_hazardous':       True,
            'horizon_condition':  horizon_condition,
            'verbose':            False,
            'max_cards_per_draw': max_cards_per_draw,
        })
        pomdp = POMDPFactory('vanilla')
        pomdp.__init__(**p)
        pomdp.value_iteration()
        return pomdp

    pomdp_short = _make_pomdp('short')
    pomdp_long  = _make_pomdp('long')

    plot_yellow_blue_fractions_separate(
        best_actions_short=pomdp_short.best_actions,
        best_actions_long=pomdp_long.best_actions,
        max_cards_per_draw=max_cards_per_draw,
        counts_dict_yellow_short=counts_dict_yellow_short,
        counts_dict_blue_short=counts_dict_blue_short,
        counts_dict_yellow_long=counts_dict_yellow_long,
        counts_dict_blue_long=counts_dict_blue_long,
        path=figure_path,
        ytick_step_short=ytick_step_short,
        ytick_step_long=ytick_step_long,
        ylim_long=30, ylim_short=30,
        dark=dark, 
        font_size=font_size,
    )

run_plot_yellow_blue_separate(
    counts_dict_yellow_short, counts_dict_blue_short,
    counts_dict_yellow_long,  counts_dict_blue_long,
    dark=True
)


In [ ]:
# from src.utils.plotting import plot_yellow_blue_fractions_separate

# plot_yellow_blue_fractions_separate(
#     best_actions_short=pomdp_short.best_actions,
#     best_actions_long=pomdp_long.best_actions,
#     max_cards_per_draw=max_cards_per_draw,
#     counts_dict_yellow_short=counts_dict_yellow_short,
#     counts_dict_blue_short=counts_dict_blue_short,
#     counts_dict_yellow_long=counts_dict_yellow_long,
#     counts_dict_blue_long=counts_dict_blue_long,
#     path=figure_path,
#     ylim_long=30, ylim_short=30,
#     dark=False, bg_color='white',
#     font_size=26,
# )


In [ ]:
# upload the ocr.csv file 
ocir_file_path = os.path.join(project_root, "data/TrHu_NHB_light/data_MEG/fa_scores.csv")
ybocs_path=os.path.join(project_root,"data/TrHu_NHB_light/data_MEG/ybocs_scores.csv")
ocir_data = pd.read_csv(ocir_file_path)
ybocs_data=pd.read_csv(ybocs_path)
# obtain the userID when the other row values are not non
ocd_userID=ybocs_data[ybocs_data.drop(columns='userID').notna().all(axis=1)]['userID']
non_ocd_userID=ybocs_data[ybocs_data.drop(columns=['userID']).isna().all(axis=1)]['userID']

In [ ]:
counts=analyze_meg_draws(human_meg_data,userID_list=ocd_userID)
pair_count_short=counts['pair_count_short']
results_ocd=counts['results']
pair_count_long=counts['pair_count_long']
counts_dict_short=counts['counts_dict_short']
counts_dict_long=counts['counts_dict_long']
num_games=counts['num_games']
run_plot_action_values(counts_dict_short,horizon_condition='short')
run_plot_action_values(counts_dict_long,horizon_condition='long')

In [ ]:
counts=analyze_meg_draws(human_meg_data,userID_list=non_ocd_userID)
results_non_ocd=counts['results']
pair_count_short=counts['pair_count_short']
pair_count_long=counts['pair_count_long']
counts_dict_short=counts['counts_dict_short']
counts_dict_long=counts['counts_dict_long']
num_games=counts['num_games']
run_plot_action_values(counts_dict_short,horizon_condition='short')
run_plot_action_values(counts_dict_long,horizon_condition='long')

In [ ]:
counts=analyze_meg_draws(human_meg_data,userID_list=[28])
pair_count_short=counts['pair_count_short']
results_ocd=counts['results']
pair_count_long=counts['pair_count_long']
counts_dict_short=counts['counts_dict_short']
counts_dict_long=counts['counts_dict_long']
num_games=counts['num_games']
run_plot_action_values(counts_dict_short,horizon_condition='short')
run_plot_action_values(counts_dict_long,horizon_condition='long')

In [ ]:
num_draws_both_all_groups,num_draws_long_all_groups,num_draws_short_all_groups=calc_num_draws(results)
plot_num_draws(num_draws_both_all_groups,num_draws_long_all_groups,num_draws_short_all_groups)


In [ ]:
num_draws_both_ocd,num_draws_long_ocd,num_draws_short_ocd=calc_num_draws(results_ocd)
plot_num_draws(num_draws_both_ocd,num_draws_long_ocd,num_draws_short_ocd)


In [ ]:
num_draws_both_healthy,num_draws_long_healthy,num_draws_short_healthy=calc_num_draws(results_non_ocd)
plot_num_draws(num_draws_both_healthy,num_draws_long_healthy,num_draws_short_healthy)


In [ ]:
def plot_all_num_draws(num_draws_all, num_draws_healthy, num_draws_ocd):
    """
    num_draws_all/healthy/ocd: tuples of (num_draws_both, num_draws_long, num_draws_short)
    """
    fig, axes = plt.subplots(3, 3, figsize=(15, 10), sharex='col', sharey='row')

    groups = ['All Participants', 'Non-OCD Group', 'OCD Group']
    draw_types = ['Short Sequence', 'Long Sequence', 'Both Sequences']
    data_groups = [num_draws_all, num_draws_healthy, num_draws_ocd]

    for row in range(3):
        draws_both, draws_long, draws_short = data_groups[row]

        sns.histplot(draws_short, bins=8, kde=True, color='skyblue', ax=axes[row][0])
        sns.histplot(draws_long, bins=14, kde=True, color='skyblue', ax=axes[row][1])
        sns.histplot(draws_both, bins=14, kde=True, color='skyblue', ax=axes[row][2])

        for col in range(3):
            ax = axes[row][col]
            xmin, xmax = ax.get_xlim()
            ax.set_xticks(np.arange(int(np.floor(xmin)), int(np.ceil(xmax)) + 1, 2))
            ax.set_xlabel('Number of Draws')
            ax.set_ylabel('Frequency')
            
            if row == 0:
                axes[row][col].set_title(draw_types[col])
            if col == 0:
                axes[row][col].set_ylabel(f'{groups[row]}\nFrequency')

    plt.tight_layout()
    plt.show()

# Grouped as: (num_draws_both, num_draws_long, num_draws_short)
num_draws_all = (num_draws_both_all_groups, num_draws_long_all_groups, num_draws_short_all_groups)
num_draws_healthy = (num_draws_both_healthy, num_draws_long_healthy, num_draws_short_healthy)
num_draws_ocd = (num_draws_both_ocd, num_draws_long_ocd, num_draws_short_ocd)

plot_all_num_draws(num_draws_all, num_draws_healthy, num_draws_ocd)


## OCD vs Healthy: Number of Draws and Accuracy by Horizon

In [ ]:
def compute_decision_stats(human_meg_data, userID_list, group_name=None, return_raw=False):
    """
    Mirrors the logic used for `decision_stats_df` above, but also tracks the
    number of draws taken before a decision, and accepts an arbitrary
    `userID_list` so it can be applied to subgroups (e.g. OCD vs non-OCD).
    Reports rates (not raw counts) for correct/incorrect/missed, since groups
    differ in size and raw counts would otherwise just track group size.
    If `return_raw=True`, also returns the underlying per-horizon stats dict
    (correct/incorrect/missed counts and draw lists) for significance testing.
    """
    stats = {'short': {'correct': 0, 'incorrect': 0, 'missed': 0, 'draws': []},
              'long':  {'correct': 0, 'incorrect': 0, 'missed': 0, 'draws': []}}

    n_subjects = 0
    for user_id in userID_list:
        subj = get_subject_data(human_meg_data, user_id)
        if subj is None:
            continue
        n_subjects += 1
        for (block, game), game_data in subj.groupby(['block', 'game']):
            horizon = 'short' if len(game_data) <= 8 else 'long'

            choice_col = game_data['choiceTrial']
            if choice_col.isnull().all():
                num_draws = len(game_data)
            else:
                first_choice_idx = choice_col.first_valid_index()
                pos = list(choice_col.index).index(first_choice_idx) + 1
                num_draws = pos + 1
            stats[horizon]['draws'].append(num_draws)

            reward_series = game_data['reward'].dropna()
            if reward_series.empty:
                stats[horizon]['missed'] += 1
                continue
            r = int(reward_series.iloc[-1])
            if r == 2:
                stats[horizon]['correct'] += 1
            elif r == -2:
                stats[horizon]['incorrect'] += 1
            else:
                stats[horizon]['missed'] += 1

    rows = []
    for cond in ('short', 'long'):
        s = stats[cond]
        total      = s['correct'] + s['incorrect'] + s['missed']
        acc        = s['correct']   / total if total > 0 else float('nan')
        inc_r      = s['incorrect'] / total if total > 0 else float('nan')
        miss_r     = s['missed']    / total if total > 0 else float('nan')
        mean_draws = np.mean(s['draws']) if s['draws'] else float('nan')
        rows.append({
            'Condition':           cond.capitalize(),
            'N Subjects':          n_subjects,
            'Total Trials':        total,
            'Accuracy':            f"{acc:.3f}",
            'Incorrect Rate':      f"{inc_r:.3f}",
            'Missing Rate':        f"{miss_r:.3f}",
            'Mean Num Draws':      f"{mean_draws:.3f}",
        })

    df = pd.DataFrame(rows).set_index('Condition')
    if group_name is not None:
        df.insert(0, 'Group', group_name)

    if return_raw:
        return df, stats
    return df


decision_stats_ocd_df,     raw_stats_ocd     = compute_decision_stats(human_meg_data, ocd_userID,     group_name='OCD',     return_raw=True)
decision_stats_non_ocd_df, raw_stats_non_ocd = compute_decision_stats(human_meg_data, non_ocd_userID, group_name='Non-OCD', return_raw=True)

decision_stats_by_group_df = pd.concat([decision_stats_ocd_df, decision_stats_non_ocd_df])
display(decision_stats_by_group_df)

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu

def compare_groups_by_horizon(raw_a, raw_b, label_a='OCD', label_b='Non-OCD'):
    """
    For each horizon, tests whether accuracy and missing rate differ between
    two groups (chi-square test of independence) and whether the number of
    draws differs (Mann-Whitney U test on the draw distributions).
    """
    rows = []
    for cond in ('short', 'long'):
        sa, sb = raw_a[cond], raw_b[cond]

        correct_a, total_a = sa['correct'], sa['correct'] + sa['incorrect'] + sa['missed']
        correct_b, total_b = sb['correct'], sb['correct'] + sb['incorrect'] + sb['missed']
        contingency_acc = [
            [correct_a, total_a - correct_a],
            [correct_b, total_b - correct_b],
        ]
        chi2_acc, p_acc, _, _ = chi2_contingency(contingency_acc)

        missed_a, missed_b = sa['missed'], sb['missed']
        contingency_miss = [
            [missed_a, total_a - missed_a],
            [missed_b, total_b - missed_b],
        ]
        chi2_miss, p_miss, _, _ = chi2_contingency(contingency_miss)

        u_stat, p_draws = mannwhitneyu(sa['draws'], sb['draws'], alternative='two-sided')

        rows.append({
            'Horizon':                       cond.capitalize(),
            f'Accuracy ({label_a})':         f"{correct_a / total_a:.3f}",
            f'Accuracy ({label_b})':         f"{correct_b / total_b:.3f}",
            'Accuracy p-value (chi2)':       f"{p_acc:.4f}",
            f'Missing Rate ({label_a})':     f"{missed_a / total_a:.3f}",
            f'Missing Rate ({label_b})':     f"{missed_b / total_b:.3f}",
            'Missing Rate p-value (chi2)':   f"{p_miss:.4f}",
            f'Mean Draws ({label_a})':       f"{np.mean(sa['draws']):.3f}",
            f'Mean Draws ({label_b})':       f"{np.mean(sb['draws']):.3f}",
            'Draws p-value (Mann-Whitney)':  f"{p_draws:.4f}",
        })
    return pd.DataFrame(rows).set_index('Horizon')


group_comparison_pvalues_df = compare_groups_by_horizon(raw_stats_ocd, raw_stats_non_ocd)
display(group_comparison_pvalues_df)

In [ ]:
latex_group_comparison = group_comparison_pvalues_df.reset_index().to_latex(
    index=False,
    caption="OCD vs.\\ Non-OCD comparison of accuracy, missing rate, and number of draws, by horizon. "
            "Accuracy and missing rate compared via a $\\chi^2$ test of independence; "
            "number of draws compared via a Mann--Whitney $U$ test.",
    label="tab:ocd_vs_nonocd_pvalues",
    column_format="lrrrrrrrrr",
)
print(latex_group_comparison)

## Low vs. High Compulsion (OCIR) Groups vs. Non-OCD

In [ ]:
# Composite "compulsion" score = sum of the compulsion-related OCI-R subscales
# (Washing, Checking, Ordering, Neutralizing) -- excludes Obsessing (an obsession,
# not compulsion, subscale) and Hoarding (its own separate construct).
ocir_data['compulsion_score'] = ocir_data[
    ['OCIR_Washing', 'OCIR_Checking', 'OCIR_Ordering', 'OCIR_Neutralizing']
].sum(axis=1)

# Take the 20 lowest- and 20 highest-scoring subjects on this composite score,
# leaving the middle of the distribution out of the comparison entirely.
N_PER_GROUP = 20
ocir_sorted_by_compulsion = ocir_data.sort_values('compulsion_score')
low_compulsion_userID  = ocir_sorted_by_compulsion.iloc[:N_PER_GROUP]['userID']
high_compulsion_userID = ocir_sorted_by_compulsion.iloc[-N_PER_GROUP:]['userID']

print(f"Low compulsion:  n = {len(low_compulsion_userID)}, "
      f"compulsion_score in [{ocir_sorted_by_compulsion['compulsion_score'].iloc[0]:.0f}, "
      f"{ocir_sorted_by_compulsion['compulsion_score'].iloc[N_PER_GROUP - 1]:.0f}]")
print(f"High compulsion: n = {len(high_compulsion_userID)}, "
      f"compulsion_score in [{ocir_sorted_by_compulsion['compulsion_score'].iloc[-N_PER_GROUP]:.0f}, "
      f"{ocir_sorted_by_compulsion['compulsion_score'].iloc[-1]:.0f}]")

decision_stats_low_df,  raw_stats_low  = compute_decision_stats(human_meg_data, low_compulsion_userID,  group_name='Low Compulsion',  return_raw=True)
decision_stats_high_df, raw_stats_high = compute_decision_stats(human_meg_data, high_compulsion_userID, group_name='High Compulsion', return_raw=True)

decision_stats_compulsion_df = pd.concat([decision_stats_low_df, decision_stats_high_df, decision_stats_non_ocd_df])
display(decision_stats_compulsion_df)

In [ ]:
low_vs_normal_pvalues_df  = compare_groups_by_horizon(raw_stats_low,  raw_stats_non_ocd, label_a='Low Compulsion',  label_b='Non-OCD')
high_vs_normal_pvalues_df = compare_groups_by_horizon(raw_stats_high, raw_stats_non_ocd, label_a='High Compulsion', label_b='Non-OCD')

print("Low Compulsion vs. Non-OCD:")
display(low_vs_normal_pvalues_df)

print("High Compulsion vs. Non-OCD:")
display(high_vs_normal_pvalues_df)


In [ ]:
latex_low_vs_normal = low_vs_normal_pvalues_df.reset_index().to_latex(
    index=False,
    caption="Low-compulsion (20 lowest scorers on Washing+Checking+Ordering+Neutralizing) "
            "vs.\\ Non-OCD comparison of accuracy, missing rate, and number of draws, by horizon.",
    label="tab:low_compulsion_vs_nonocd",
    column_format="lrrrrrrrrr",
)
print(latex_low_vs_normal)

latex_high_vs_normal = high_vs_normal_pvalues_df.reset_index().to_latex(
    index=False,
    caption="High-compulsion (20 highest scorers on Washing+Checking+Ordering+Neutralizing) "
            "vs.\\ Non-OCD comparison of accuracy, missing rate, and number of draws, by horizon.",
    label="tab:high_compulsion_vs_nonocd",
    column_format="lrrrrrrrrr",
)
print(latex_high_vs_normal)

### High Compulsion vs. All Other Subjects (not just Non-OCD)

In [ ]:
# Compare the High Compulsion group against everyone else in the sample
# (i.e. all 85 remaining subjects, not just the Non-OCD subset).
all_other_userID = pd.Series([uid for uid in userID_list if uid not in set(high_compulsion_userID)])

decision_stats_rest_df, raw_stats_rest = compute_decision_stats(human_meg_data, all_other_userID, group_name='All Other', return_raw=True)

decision_stats_high_vs_rest_df = pd.concat([decision_stats_high_df, decision_stats_rest_df])
display(decision_stats_high_vs_rest_df)


In [ ]:
high_vs_rest_pvalues_df = compare_groups_by_horizon(raw_stats_high, raw_stats_rest, label_a='High Compulsion', label_b='All Other')
display(high_vs_rest_pvalues_df)


In [ ]:
latex_high_vs_rest = high_vs_rest_pvalues_df.reset_index().to_latex(
    index=False,
    caption="High-compulsion (20 highest scorers on Washing+Checking+Ordering+Neutralizing) "
            "vs.\\ all other subjects: comparison of accuracy, missing rate, and number of draws, by horizon.",
    label="tab:high_compulsion_vs_rest",
    column_format="lrrrrrrrrr",
)
print(latex_high_vs_rest)


## YBOCS / OCIR Metrics vs Number of Draws, by Horizon

In [ ]:
def compute_per_subject_metrics(human_meg_data, userID_list):
    """
    Per-subject mean number of draws and accuracy, computed separately for the
    short- and long-horizon conditions, for use in scatter plots against
    YBOCS / OCIR-R scores.
    """
    records = []
    for user_id in userID_list:
        subj = get_subject_data(human_meg_data, user_id)
        if subj is None:
            continue

        draws   = {'short': [], 'long': []}
        correct = {'short': 0, 'long': 0}
        total   = {'short': 0, 'long': 0}

        for (block, game), game_data in subj.groupby(['block', 'game']):
            horizon = 'short' if len(game_data) <= 8 else 'long'

            choice_col = game_data['choiceTrial']
            if choice_col.isnull().all():
                num_draws = len(game_data)
            else:
                first_choice_idx = choice_col.first_valid_index()
                pos = list(choice_col.index).index(first_choice_idx) + 1
                num_draws = pos + 1
            draws[horizon].append(num_draws)

            total[horizon] += 1
            reward_series = game_data['reward'].dropna()
            if not reward_series.empty and int(reward_series.iloc[-1]) == 2:
                correct[horizon] += 1

        records.append({
            'userID':           user_id,
            'mean_draws_short': np.mean(draws['short']) if draws['short'] else np.nan,
            'mean_draws_long':  np.mean(draws['long'])  if draws['long']  else np.nan,
            'accuracy_short':   correct['short'] / total['short'] if total['short'] else np.nan,
            'accuracy_long':    correct['long']  / total['long']  if total['long']  else np.nan,
        })

    return pd.DataFrame(records)


per_subject_df = compute_per_subject_metrics(human_meg_data, userID_list)
per_subject_df['userID_str'] = per_subject_df['userID'].astype(str)

ybocs_data['userID_str'] = ybocs_data['userID'].astype(str)
ocir_data['userID_str']  = ocir_data['userID'].astype(str)

merged_ybocs_df = pd.merge(per_subject_df, ybocs_data, on='userID_str', how='inner')
merged_ocir_df  = pd.merge(per_subject_df, ocir_data,  on='userID_str', how='inner')

ybocs_metrics = [c for c in ybocs_data.columns if c.startswith('YBOCS_')]
ocir_metrics  = [c for c in ocir_data.columns if c.startswith('OCIR_')]


In [ ]:
from scipy.stats import pearsonr

def plot_metric_vs_draws(df, metric_col, draws_col, horizon_label, color='steelblue', save_fig=False,
                          title_prefix=None, simple_xlabel=False):
    _set_plot_style()

    sub = df[[metric_col, draws_col]].dropna()
    if len(sub) < 3:
        print(f"Skipping {metric_col} vs {draws_col}: insufficient data (n={len(sub)})")
        return

    corr, p_value = pearsonr(sub[draws_col], sub[metric_col])
    sig_label = "*" if p_value < 0.05 else "n.s."
    metric_label = metric_col.replace('_', r'\_') if plt.rcParams.get('text.usetex') else metric_col
    title_label = title_prefix if title_prefix is not None else metric_label

    fig = plt.figure(figsize=(7, 5))
    plt.scatter(sub[draws_col], sub[metric_col], c=color, alpha=0.6, s=80,
                edgecolors="white", linewidth=0.5)
    sns.regplot(data=sub, x=draws_col, y=metric_col, scatter=False, color="black",
                line_kws={"linestyle": "--", "linewidth": 2})
    plt.title(f"{title_label} vs Number of Draws ({horizon_label} Horizon)\n"
              f"r = {corr:.3f} ({sig_label}, p = {p_value:.3f}, n = {len(sub)})")
    plt.xlabel("Mean Number of Draws" if simple_xlabel else f"Mean Number of Draws ({horizon_label})")
    plt.ylabel(metric_label)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_fig:
        save_figure(f"{metric_col}_vs_draws_{horizon_label.lower()}.png")
    plt.show()

### YBOCS metrics vs draws (OCD subjects only — non-OCD have missing YBOCS)

In [ ]:
for metric in ybocs_metrics:
    plot_metric_vs_draws(merged_ybocs_df, metric, 'mean_draws_short', 'Short')
    plot_metric_vs_draws(merged_ybocs_df, metric, 'mean_draws_long',  'Long')


### Beamer summary: YBOCS vs draws (key metrics, with switched-behavior subjects flagged)


In [ ]:
def flag_switch_subjects(df, ybocs_col='YBOCS_total_score',
                          switch_quantile=0.75, ybocs_quantile=0.5):
    """
    Flags subjects with a high YBOCS score who also show a behavioral
    'switch': relatively many draws in the long horizon but relatively
    few draws in the short horizon, compared to the rest of the sample.
    """
    z_short = (df['mean_draws_short'] - df['mean_draws_short'].mean()) / df['mean_draws_short'].std()
    z_long  = (df['mean_draws_long']  - df['mean_draws_long'].mean())  / df['mean_draws_long'].std()
    switch_score = z_long - z_short

    high_ybocs = df[ybocs_col] >= df[ybocs_col].quantile(ybocs_quantile)
    high_switch = switch_score >= switch_score.quantile(switch_quantile)

    return high_ybocs & high_switch, switch_score


switch_mask, switch_score = flag_switch_subjects(merged_ybocs_df)
merged_ybocs_df['switch_score'] = switch_score
merged_ybocs_df['is_switcher'] = switch_mask

print(f"Flagged {switch_mask.sum()} / {len(merged_ybocs_df)} subjects as high-YBOCS switchers:")
merged_ybocs_df.loc[switch_mask, ['userID', 'YBOCS_total_score', 'mean_draws_short', 'mean_draws_long', 'switch_score']]


In [ ]:
def plot_ybocs_summary_grid(df, metrics, highlight_mask=None,
                             draws_cols=('mean_draws_short', 'mean_draws_long'),
                             horizon_labels=('Short', 'Long'),
                             font_size=14, fname='ybocs_summary_grid.pdf',
                             save_fig=True):
    """
    Multi-panel beamer-ready figure: rows = horizon, cols = metric.
    Subjects in `highlight_mask` (e.g. high-YBOCS switchers) are drawn
    in red on top of the rest of the sample. Returns a DataFrame with
    the correlation stats for each panel (for the companion summary table).
    """
    _set_plot_style(font_size=font_size)
    n_rows, n_cols = len(draws_cols), len(metrics)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.2 * n_cols, 4.2 * n_rows))
    axes = np.atleast_2d(axes)

    rows = []
    for i, (draws_col, horizon_label) in enumerate(zip(draws_cols, horizon_labels)):
        for j, metric in enumerate(metrics):
            ax = axes[i, j]
            sub = df[[metric, draws_col]].dropna()
            corr, p_value = pearsonr(sub[draws_col], sub[metric])
            sig_label = "*" if p_value < 0.05 else "n.s."

            base = sub if highlight_mask is None else sub[~highlight_mask.reindex(sub.index, fill_value=False)]
            ax.scatter(base[draws_col], base[metric], c='steelblue', alpha=0.6, s=50,
                       edgecolors='white', linewidth=0.5, label='Other subjects')

            if highlight_mask is not None:
                flagged = sub[highlight_mask.reindex(sub.index, fill_value=False)]
                if not flagged.empty:
                    ax.scatter(flagged[draws_col], flagged[metric], c='crimson', alpha=0.9, s=70,
                               edgecolors='black', linewidth=0.6, marker='D', label='High-YBOCS switchers')

            sns.regplot(data=sub, x=draws_col, y=metric, scatter=False, color='black',
                        line_kws={'linestyle': '--', 'linewidth': 1.5}, ax=ax)

            ax.set_title(f"r={corr:.2f} ({sig_label}, p={p_value:.3f})", fontsize=font_size - 2)
            ax.set_xlabel(f"Draws ({horizon_label})")
            ax.set_ylabel(metric.replace('YBOCS_', '') if j == 0 else '')
            ax.grid(True, alpha=0.3)

            rows.append({
                'metric': metric, 'horizon': horizon_label,
                'r': corr, 'p_value': p_value, 'n': len(sub),
            })

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.04), frameon=False)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    if save_fig:
        save_figure(fname)
    plt.show()

    return pd.DataFrame(rows)


In [ ]:
key_ybocs_metrics = ['YBOCS_total_score', 'YBOCS_insight', 'YBOCS_Indecisiveness', 'YBOCS_compulsions_subtotal']

ybocs_summary_stats_df = plot_ybocs_summary_grid(
    merged_ybocs_df, key_ybocs_metrics, highlight_mask=merged_ybocs_df['is_switcher'],
    fname='ybocs_summary_grid.pdf',
)

latex_summary = ybocs_summary_stats_df.round({'r': 3, 'p_value': 3}).to_latex(
    index=False,
    caption='Correlation between YBOCS metrics and mean number of draws, by horizon.',
    label='tab:ybocs_summary',
    column_format='llrrr',
)
print(latex_summary)
ybocs_summary_stats_df


### OCIR metrics vs draws (all subjects)

In [ ]:
for metric in ocir_metrics:
    plot_metric_vs_draws(merged_ocir_df, metric, 'mean_draws_short', 'Short')
    plot_metric_vs_draws(merged_ocir_df, metric, 'mean_draws_long',  'Long')


### Compulsion-related OCIR subscales vs draws (all subjects, not split by group)

In [ ]:
compulsion_metrics = ['OCIR_Washing', 'OCIR_Checking', 'OCIR_Ordering', 'OCIR_Neutralizing']

for metric in compulsion_metrics:
    plot_metric_vs_draws(merged_ocir_df, metric, 'mean_draws_short', 'Short')
    plot_metric_vs_draws(merged_ocir_df, metric, 'mean_draws_long',  'Long')


### Combined compulsion score (Washing + Checking + Ordering + Neutralizing) vs draws (all subjects)

In [ ]:
plot_metric_vs_draws(merged_ocir_df, 'compulsion_score', 'mean_draws_short', 'Short', save_fig=True)
plot_metric_vs_draws(merged_ocir_df, 'compulsion_score', 'mean_draws_long',  'Long',  save_fig=True)

### Compulsion score vs draws, within the Low and High compulsion groups separately

In [ ]:
merged_ocir_low_df  = merged_ocir_df[merged_ocir_df['userID_str'].isin(low_compulsion_userID.astype(str))]
merged_ocir_high_df = merged_ocir_df[merged_ocir_df['userID_str'].isin(high_compulsion_userID.astype(str))]

print("Low Compulsion group:")
plot_metric_vs_draws(merged_ocir_low_df, 'compulsion_score', 'mean_draws_short', 'Short',
                      title_prefix='Low Compulsion', simple_xlabel=True)
plot_metric_vs_draws(merged_ocir_low_df, 'compulsion_score', 'mean_draws_long',  'Long',
                      title_prefix='Low Compulsion', simple_xlabel=True)

print("High Compulsion group:")
plot_metric_vs_draws(merged_ocir_high_df, 'compulsion_score', 'mean_draws_short', 'Short',
                      title_prefix='High Compulsion', simple_xlabel=True)
plot_metric_vs_draws(merged_ocir_high_df, 'compulsion_score', 'mean_draws_long',  'Long',
                      title_prefix='High Compulsion', simple_xlabel=True)

In [ ]:
# plot the psychometric curve of the human data for each subject. 

all_evidence_path = os.path.join(
            project_root, f"data/TrHu_NHB_light/data_MEG/all_subject_evidence_dicts.pkl"
        )

In [ ]:
all_subject_evidence_dicts_of_dict = pd.read_pickle(all_evidence_path)

In [ ]:
# get the first subject by iloc[0]
all_subject_evidence_dicts_of_dict.iloc[0]
# there are two horizons, plot the psychometric curve separately then combined. 
# long horizon 
long_horizon_data=all_subject_evidence_dicts_of_dict.iloc[0]['long']

# now each row is one game, it consisits list of lists, where each sublist is a trial of current draw, yellow, blue, action, and outcome. The outcome is correct=2, incorrect=-2, and missing is -1
# The psychometric curve is that, the proportion of the correct choice as a function of yellow-blue. 

# 1) loop over the rows and extract only the last trial for each row, calculate the yellow-blue and the outcome.
psychometric_curve={}
for index, row in long_horizon_data.iterrows():
    draw,yellow,blue,action,outcome=(row['draw_yellow_blue_action_outcome'][-1])
    ev_diff=yellow-blue
    #check if the key in the dictionary 
    
    

In [ ]:
# get the first subject by iloc[0]
subject_id=28
all_subject_evidence_dicts_of_dict['long'][subject_id]
    
    

In [ ]:
import matplotlib.colors as mcolors
import matplotlib as mpl
colors = ["lightgray", "#3FD24B", "#E92424"]


# 2. Create the custom colormap
cmap = mcolors.LinearSegmentedColormap.from_list("RedGreyBlack", colors)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# **MERGE ALL SUBJECTS** - Include missing data as 0.5
all_ev_diffs_long = []
all_outcomes_long = []
all_ev_diffs_short = []
all_outcomes_short = []

for subject_idx in range(len(all_subject_evidence_dicts_of_dict)):
    subject_data = all_subject_evidence_dicts_of_dict.iloc[subject_idx]
    
    # LONG horizon - all subjects
    long_horizon_data = subject_data['long']
    for index, row in long_horizon_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        
        if outcome == 2:  # correct
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(1)
        elif outcome == -2:  # incorrect
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(0)
        elif outcome == -1:  # missing
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(0.5)

    # SHORT horizon - all subjects
    if 'short' in subject_data:
        short_horizon_data = subject_data['short']
        for index, row in short_horizon_data.iterrows():
            last_trial = row['draw_yellow_blue_action_outcome'][-1]
            draw, yellow, blue, action, outcome = last_trial
            
            ev_diff = yellow - blue
            
            if outcome == 2:
                all_ev_diffs_short.append(ev_diff)
                all_outcomes_short.append(1)
            elif outcome == -2:
                all_ev_diffs_short.append(ev_diff)
                all_outcomes_short.append(0)
            elif outcome == -1:
                all_ev_diffs_short.append(ev_diff)
                all_outcomes_short.append(0.5)

# **PLOT LONG HORIZON - 0 to 1 scale**
plt.figure(figsize=(8, 6))
bins_long = np.linspace(min(all_ev_diffs_long), max(all_ev_diffs_long), 12)
bin_centers_long = (bins_long[:-1] + bins_long[1:]) / 2

prop_correct_long = []
for i in range(len(bins_long)-1):
    mask = (np.array(all_ev_diffs_long) >= bins_long[i]) & (np.array(all_ev_diffs_long) < bins_long[i+1])
    bin_outcomes = np.array(all_outcomes_long)[mask]
    prop_correct_long.append(np.mean(bin_outcomes) if len(bin_outcomes) > 0 else np.nan)

valid_long = ~np.isnan(prop_correct_long)
plt.scatter(bin_centers_long[valid_long], np.array(prop_correct_long)[valid_long], 
            color='green', s=120, alpha=0.8, edgecolors='darkblue', linewidth=1.8)
plt.xlabel('Yellow - Blue Evidence Difference')
plt.ylabel('Proportion Correct')
plt.title(f'Psychometric Curve - LONG Horizon\n(All Subjects, N={len(all_ev_diffs_long)} trials)')
plt.grid(True, alpha=0.3)
plt.ylim(0, 1.2)
plt.tight_layout()
plt.show()

# **PLOT SHORT HORIZON - 0 to 1 scale**
if len(all_ev_diffs_short) > 0:
    plt.figure(figsize=(8, 6))
    
    bins_short = np.linspace(min(all_ev_diffs_short), max(all_ev_diffs_short), 12)
    bin_centers_short = (bins_short[:-1] + bins_short[1:]) / 2
    
    prop_correct_short = []
    for i in range(len(bins_short)-1):
        mask = (np.array(all_ev_diffs_short) >= bins_short[i]) & (np.array(all_ev_diffs_short) < bins_short[i+1])
        bin_outcomes = np.array(all_outcomes_short)[mask]
        prop_correct_short.append(np.mean(bin_outcomes) if len(bin_outcomes) > 0 else np.nan)
    
    valid_short = ~np.isnan(prop_correct_short)
    plt.scatter(bin_centers_short[valid_short], np.array(prop_correct_short)[valid_short], 
                color='green', s=120, alpha=0.8, edgecolors='darkblue', linewidth=1.8)
    plt.xlabel('Yellow - Blue Evidence Difference')
    plt.ylabel('Proportion Correct')
    plt.title(f'Psychometric Curve - SHORT Horizon\n(All Subjects, N={len(all_ev_diffs_short)} trials)')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1.2)
    plt.tight_layout()
    plt.show()

# Summary stats
print(f"Long horizon: {len(all_ev_diffs_long)} total trials ({sum(1 for x in all_outcomes_long if x==1)} correct, {sum(1 for x in all_outcomes_long if x==0)} incorrect, {sum(1 for x in all_outcomes_long if x==0.5)} missing)")
print(f"  Mean performance: {np.mean(all_outcomes_long):.3f}")
if len(all_ev_diffs_short) > 0:
    print(f"Short horizon: {len(all_ev_diffs_short)} total trials ({sum(1 for x in all_outcomes_short if x==1)} correct, {sum(1 for x in all_outcomes_short if x==0)} incorrect, {sum(1 for x in all_outcomes_short if x==0.5)} missing)")
    print(f"  Mean performance: {np.mean(all_outcomes_short):.3f}")


In [ ]:
def plot_psychometric_curve(subject_id, horizon='long', include_missing=True):
    """
    Plot psychometric curve for specific subject - SAME LOGIC as all-subjects code
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Get subject data - YOUR ACCESS METHOD
    subject_data = all_subject_evidence_dicts_of_dict[horizon][subject_id]
    
    # Extract data - IDENTICAL LOGIC
    ev_diffs = []
    outcomes = []
    
    for index, row in subject_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        
        if outcome == 2:  # correct
            ev_diffs.append(ev_diff)
            outcomes.append(1.0)
        elif outcome == -2:  # incorrect
            ev_diffs.append(ev_diff)
            outcomes.append(0.0)
        elif outcome == -1 and include_missing:  # missing as 0.5
            ev_diffs.append(ev_diff)
            outcomes.append(0.5)
    
    ev_diffs = np.array(ev_diffs)
    outcomes = np.array(outcomes)
    
    # Binning - IDENTICAL LOGIC
    bins = np.linspace(ev_diffs.min(), ev_diffs.max(), 12)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    prop_correct = []
    for i in range(len(bins)-1):
        # FIXED: Same mask logic as your code
        mask = (ev_diffs >= bins[i]) & (ev_diffs < bins[i+1])
        bin_outcomes = outcomes[mask]
        prop_correct.append(np.mean(bin_outcomes) if len(bin_outcomes) > 0 else np.nan)
    
    # Plotting - IDENTICAL
    valid = ~np.isnan(prop_correct)
    plt.figure(figsize=(8, 6))
    plt.scatter(bin_centers[valid], np.array(prop_correct)[valid], 
                color='green', s=120, alpha=0.8, edgecolors='darkblue', linewidth=1.8)
    plt.xlabel('Yellow - Blue Evidence Difference')
    plt.ylabel('Proportion Correct')
    plt.title(f'Psychometric Curve - Subject {subject_id} ({horizon.upper()})\nN={len(ev_diffs)} trials')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1.2)
    plt.tight_layout()
    plt.show()
    
    # Summary - IDENTICAL
    n_correct = sum(outcomes == 1.0)
    n_incorrect = sum(outcomes == 0.0)
    n_missing = sum(outcomes == 0.5)
    print(f"Subject {subject_id} ({horizon.upper()}): {len(ev_diffs)} trials")
    print(f"  Correct: {n_correct}, Incorrect: {n_incorrect}, Missing: {n_missing}")
    print(f"  Mean: {np.mean(outcomes):.3f}")

# Usage
subject_id=17
plot_psychometric_curve(subject_id, 'long')
plot_psychometric_curve(subject_id, 'short')


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

n_bins = 12
all_ev_diffs_long = []
all_outcomes_long = []
all_ev_diffs_short = []
all_outcomes_short = []

for subject_idx in range(len(all_subject_evidence_dicts_of_dict)):
    subject_data = all_subject_evidence_dicts_of_dict.iloc[subject_idx]
    
    # ---------- LONG horizon ----------
    long_horizon_data = subject_data['long']
    for index, row in long_horizon_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        # define whether yellow is the correct color
        if yellow > blue:
            yellow_correct = 1
        elif yellow < blue:
            yellow_correct = 0
        else:
            yellow_correct = 0.5
        
        if outcome == -1:  # missing trial
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(0.5)  # fixed from 0.25
        else:
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(yellow_correct)

    # ---------- SHORT horizon ----------
    if 'short' in subject_data:
        short_horizon_data = subject_data['short']
        for index, row in short_horizon_data.iterrows():
            last_trial = row['draw_yellow_blue_action_outcome'][-1]
            draw, yellow, blue, action, outcome = last_trial
            
            ev_diff = yellow - blue
            if yellow > blue:
                yellow_correct = 1
            elif yellow < blue:
                yellow_correct = 0
            else:
                yellow_correct = 0.5
            
            # if outcome == -1:  # missing trial
            #     all_ev_diffs_short.append(ev_diff)
            #     all_outcomes_short.append(0.5)  # fixed from 0.25
            all_ev_diffs_short.append(ev_diff)
            all_outcomes_short.append(yellow_correct)

# ---------- PLOT LONG HORIZON WITH COLORBAR ----------
plt.figure(figsize=(10, 6))
all_ev_diffs_long_arr = np.array(all_ev_diffs_long)
all_outcomes_long_arr = np.array(all_outcomes_long)

bins_long = np.linspace(all_ev_diffs_long_arr.min(), all_ev_diffs_long_arr.max(), n_bins)
bin_centers_long = (bins_long[:-1] + bins_long[1:]) / 2

prop_correct_long = []
bin_counts_long = []
for i in range(len(bins_long) - 1):
    mask = (all_ev_diffs_long_arr >= bins_long[i]) & (all_ev_diffs_long_arr < bins_long[i + 1])
    bin_outcomes = all_outcomes_long_arr[mask]
    bin_count = len(bin_outcomes)
    if bin_count > 0:
        prop_correct_long.append(np.mean(bin_outcomes))
        bin_counts_long.append(bin_count)
    else:
        prop_correct_long.append(np.nan)
        bin_counts_long.append(0)

prop_correct_long = np.array(prop_correct_long)
valid_long = ~np.isnan(prop_correct_long)

scatter = plt.scatter(
    bin_centers_long[valid_long],
    prop_correct_long[valid_long],
    c=np.array(bin_counts_long)[valid_long],  # color by trials per bin
    s=120,
    cmap=cmap,
    alpha=0.8,
    edgecolors='black',
    linewidth=1.8,
)
plt.colorbar(scatter, label='Trials per bin')
plt.xlabel('Yellow - Blue Evidence Difference')
plt.ylabel('P(Yellow is Correct)')
plt.title(f'Psychometric Curve - LONG Horizon\n(All Subjects, N={len(all_ev_diffs_long)} trials)')
plt.grid(True, alpha=0.3)
plt.ylim(-0.2, 1.2)
plt.tight_layout()
plt.show()

# ---------- PLOT SHORT HORIZON WITH COLORBAR ----------
if len(all_ev_diffs_short) > 0:
    plt.figure(figsize=(10, 6))
    all_ev_diffs_short_arr = np.array(all_ev_diffs_short)
    all_outcomes_short_arr = np.array(all_outcomes_short)
    
    bins_short = np.linspace(all_ev_diffs_short_arr.min(), all_ev_diffs_short_arr.max(), n_bins)
    bin_centers_short = (bins_short[:-1] + bins_short[1:]) / 2
    
    prop_correct_short = []
    bin_counts_short = []
    for i in range(len(bins_short) - 1):
        mask = (all_ev_diffs_short_arr >= bins_short[i]) & (all_ev_diffs_short_arr < bins_short[i + 1])
        bin_outcomes = all_outcomes_short_arr[mask]
        bin_count = len(bin_outcomes)
        if bin_count > 0:
            prop_correct_short.append(np.mean(bin_outcomes))
            bin_counts_short.append(bin_count)
        else:
            prop_correct_short.append(np.nan)
            bin_counts_short.append(0)
    
    prop_correct_short = np.array(prop_correct_short)
    valid_short = ~np.isnan(prop_correct_short)
    
    scatter = plt.scatter(
        bin_centers_short[valid_short],
        prop_correct_short[valid_short],
        c=np.array(bin_counts_short)[valid_short],  # color by trials per bin
        s=120,
        cmap=cmap,
        alpha=0.8,
        edgecolors='black',
        linewidth=1.8,
    )
    plt.colorbar(scatter, label='Trials per bin')
    plt.xlabel('Yellow - Blue Evidence Difference')
    plt.ylabel('P(Yellow is Correct)')
    plt.title(f'Psychometric Curve - SHORT Horizon\n(All Subjects, N={len(all_ev_diffs_short)} trials)')
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.20, 1.2)
    plt.tight_layout()
    plt.show()

# ---------- Summary stats ----------
print(
    f"Long horizon: {len(all_ev_diffs_long)} total trials "
    f"({sum(1 for x in all_outcomes_long if x == 1)} yellow-correct, "
    f"{sum(1 for x in all_outcomes_long if x == 0)} blue-correct, "
    f"{sum(1 for x in all_outcomes_long if x == 0.5)} missing)"
)
print(f"  Mean P(yellow correct): {np.mean(all_outcomes_long):.3f}")

if len(all_ev_diffs_short) > 0:
    print(
        f"Short horizon: {len(all_ev_diffs_short)} total trials "
        f"({sum(1 for x in all_outcomes_short if x == 1)} yellow-correct, "
        f"{sum(1 for x in all_outcomes_short if x == 0)} blue-correct, "
        f"{sum(1 for x in all_outcomes_short if x == 0.5)} missing)"
    )
    print(f"  Mean P(yellow correct): {np.mean(all_outcomes_short):.3f}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

n_bins = 12
all_ev_diffs_long = []
all_outcomes_long = []
all_ev_diffs_short = []
all_outcomes_short = []

for subject_idx in range(len(all_subject_evidence_dicts_of_dict)):
    subject_data = all_subject_evidence_dicts_of_dict.iloc[subject_idx]
    
    # ---------- LONG horizon ----------
    long_horizon_data = subject_data['long']
    for index, row in long_horizon_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        # define whether yellow is the correct color
        if yellow > blue:
            yellow_correct = 1
        elif yellow < blue:
            yellow_correct = 0
        else:
            yellow_correct = 0.5
        
        if outcome == -1:  # missing trial
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(0.5)
        else:
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(yellow_correct)

    # ---------- SHORT horizon ----------
    if 'short' in subject_data:
        short_horizon_data = subject_data['short']
        for index, row in short_horizon_data.iterrows():
            last_trial = row['draw_yellow_blue_action_outcome'][-1]
            draw, yellow, blue, action, outcome = last_trial
            
            ev_diff = yellow - blue
            if yellow > blue:
                yellow_correct = 1
            elif yellow < blue:
                yellow_correct = 0
            else:
                yellow_correct = 0.5
            
            if outcome == -1:  # missing trial
                all_ev_diffs_short.append(ev_diff)
                all_outcomes_short.append(0.5)
            else:
                all_ev_diffs_short.append(ev_diff)
                all_outcomes_short.append(yellow_correct)

# ---------- PLOT LONG HORIZON WITH COLORBAR ----------
plt.figure(figsize=(10, 6))
all_ev_diffs_long_arr = np.array(all_ev_diffs_long)
all_outcomes_long_arr = np.array(all_outcomes_long)

bins_long = np.linspace(all_ev_diffs_long_arr.min(), all_ev_diffs_long_arr.max(), n_bins)
bin_centers_long = (bins_long[:-1] + bins_long[1:]) / 2

prop_correct_long = []
bin_counts_long = []
for i in range(len(bins_long) - 1):
    mask = (all_ev_diffs_long_arr >= bins_long[i]) & (all_ev_diffs_long_arr < bins_long[i + 1])
    bin_outcomes = all_outcomes_long_arr[mask]
    bin_count = len(bin_outcomes)
    if bin_count > 0:
        prop_correct_long.append(np.mean(bin_outcomes))
        bin_counts_long.append(bin_count)
    else:
        prop_correct_long.append(np.nan)
        bin_counts_long.append(0)

prop_correct_long = np.array(prop_correct_long)
valid_long = ~np.isnan(prop_correct_long)

scatter = plt.scatter(
    bin_centers_long[valid_long],
    prop_correct_long[valid_long],
    c=np.array(bin_counts_long)[valid_long],
    s=120,
    cmap=cmap,
    alpha=0.8,
    edgecolors='black',  # Fixed: always black border
    linewidth=1.8,
)
plt.colorbar(scatter, label='Trials per bin')
plt.xlabel('Yellow - Blue Evidence Difference')
plt.ylabel('P(Yellow is Correct)')
plt.title(f'Psychometric Curve - LONG Horizon\n(All Subjects, N={len(all_ev_diffs_long)} trials)')
plt.grid(True, alpha=0.3)
plt.ylim(-0.2, 1.2)
plt.tight_layout()
plt.show()

# ---------- PLOT SHORT HORIZON WITH COLORBAR ----------
if len(all_ev_diffs_short) > 0:
    plt.figure(figsize=(10, 6))
    all_ev_diffs_short_arr = np.array(all_ev_diffs_short)
    all_outcomes_short_arr = np.array(all_outcomes_short)
    
    bins_short = np.linspace(all_ev_diffs_short_arr.min(), all_ev_diffs_short_arr.max(), n_bins)
    bin_centers_short = (bins_short[:-1] + bins_short[1:]) / 2
    
    prop_correct_short = []
    bin_counts_short = []
    for i in range(len(bins_short) - 1):
        mask = (all_ev_diffs_short_arr >= bins_short[i]) & (all_ev_diffs_short_arr < bins_short[i + 1])
        bin_outcomes = all_outcomes_short_arr[mask]
        bin_count = len(bin_outcomes)
        if bin_count > 0:
            prop_correct_short.append(np.mean(bin_outcomes))
            bin_counts_short.append(bin_count)
        else:
            prop_correct_short.append(np.nan)
            bin_counts_short.append(0)
    
    prop_correct_short = np.array(prop_correct_short)
    valid_short = ~np.isnan(prop_correct_short)
    
    scatter = plt.scatter(
        bin_centers_short[valid_short],
        prop_correct_short[valid_short],
        c=np.array(bin_counts_short)[valid_short],
        s=120,
        cmap=cmap,
        alpha=0.8,
        edgecolors='black',  # Fixed: always black border (not 'darkorange')
        linewidth=1.8,
    )
    plt.colorbar(scatter, label='Trials per bin')
    plt.xlabel('Yellow - Blue Evidence Difference')
    plt.ylabel('P(Yellow is Correct)')
    plt.title(f'Psychometric Curve - SHORT Horizon\n(All Subjects, N={len(all_ev_diffs_short)} trials)')
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.2, 1.2)
    plt.tight_layout()
    plt.show()

# ---------- Summary stats ----------
print(
    f"Long horizon: {len(all_ev_diffs_long)} total trials "
    f"({sum(1 for x in all_outcomes_long if x == 1)} yellow-correct, "
    f"{sum(1 for x in all_outcomes_long if x == 0)} blue-correct, "
    f"{sum(1 for x in all_outcomes_long if x == 0.5)} missing)"
)
print(f"  Mean P(yellow correct): {np.mean(all_outcomes_long):.3f}")

if len(all_ev_diffs_short) > 0:
    print(
        f"Short horizon: {len(all_ev_diffs_short)} total trials "
        f"({sum(1 for x in all_outcomes_short if x == 1)} yellow-correct, "
        f"{sum(1 for x in all_outcomes_short if x == 0)} blue-correct, "
        f"{sum(1 for x in all_outcomes_short if x == 0.5)} missing)"
    )
    print(f"  Mean P(yellow correct): {np.mean(all_outcomes_short):.3f}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

n_bins = 12
all_ev_diffs_long = []
all_outcomes_long = []
all_ev_diffs_short = []
all_outcomes_short = []

for subject_idx in range(len(all_subject_evidence_dicts_of_dict)):
    subject_data = all_subject_evidence_dicts_of_dict.iloc[subject_idx]
    
    # ---------- LONG horizon ----------
    long_horizon_data = subject_data['long']
    for index, row in long_horizon_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        # define whether yellow is the correct color
        if yellow > blue:
            yellow_correct = 1
        elif yellow < blue:
            yellow_correct = 0
        else:
            yellow_correct = 0.5
        
        if outcome == -1:  # missing trial
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(0.5)
        else:
            all_ev_diffs_long.append(ev_diff)
            all_outcomes_long.append(yellow_correct)

    # ---------- SHORT horizon ----------
    if 'short' in subject_data:
        short_horizon_data = subject_data['short']
        for index, row in short_horizon_data.iterrows():
            last_trial = row['draw_yellow_blue_action_outcome'][-1]
            draw, yellow, blue, action, outcome = last_trial
            
            ev_diff = yellow - blue
            if yellow > blue:
                yellow_correct = 1
            elif yellow < blue:
                yellow_correct = 0
            else:
                yellow_correct = 0.5
            
            if outcome == -1:  # missing trial
                all_ev_diffs_short.append(ev_diff)
                all_outcomes_short.append(0.5)
            else:
                all_ev_diffs_short.append(ev_diff)
                all_outcomes_short.append(yellow_correct)

# ---------- PLOT LONG HORIZON - SIZE BY TRIALS ----------
plt.figure(figsize=(10, 6))
all_ev_diffs_long_arr = np.array(all_ev_diffs_long)
all_outcomes_long_arr = np.array(all_outcomes_long)

bins_long = np.linspace(all_ev_diffs_long_arr.min(), all_ev_diffs_long_arr.max(), n_bins)
bin_centers_long = (bins_long[:-1] + bins_long[1:]) / 2

prop_correct_long = []
bin_counts_long = []
for i in range(len(bins_long) - 1):
    mask = (all_ev_diffs_long_arr >= bins_long[i]) & (all_ev_diffs_long_arr < bins_long[i + 1])
    bin_outcomes = all_outcomes_long_arr[mask]
    bin_count = len(bin_outcomes)
    if bin_count > 0:
        prop_correct_long.append(np.mean(bin_outcomes))
        bin_counts_long.append(bin_count)
    else:
        prop_correct_long.append(np.nan)
        bin_counts_long.append(0)

prop_correct_long = np.array(prop_correct_long)
valid_long = ~np.isnan(prop_correct_long)

# Scale sizes: min 30, max 300 based on trial counts
sizes_long = np.array(bin_counts_long)[valid_long]
sizes_long = 30 + (sizes_long / sizes_long.max() * 270)  # 30-300 range

plt.scatter(
    bin_centers_long[valid_long],
    prop_correct_long[valid_long],
    s=sizes_long,  # Size varies by trials per bin
    c='green',
    alpha=0.8,
    edgecolors='black',
    linewidth=1.8,
)
plt.xlabel('Yellow - Blue Evidence Difference')
plt.ylabel('P(Yellow is Correct)')
plt.title(f'Psychometric Curve - LONG Horizon\n(All Subjects, N={len(all_ev_diffs_long)} trials)')
plt.grid(True, alpha=0.3)
plt.ylim(-0.2, 1.2)
plt.tight_layout()
plt.show()

# ---------- PLOT SHORT HORIZON - SIZE BY TRIALS ----------
if len(all_ev_diffs_short) > 0:
    plt.figure(figsize=(10, 6))
    all_ev_diffs_short_arr = np.array(all_ev_diffs_short)
    all_outcomes_short_arr = np.array(all_outcomes_short)
    
    bins_short = np.linspace(all_ev_diffs_short_arr.min(), all_ev_diffs_short_arr.max(), n_bins)
    bin_centers_short = (bins_short[:-1] + bins_short[1:]) / 2
    
    prop_correct_short = []
    bin_counts_short = []
    for i in range(len(bins_short) - 1):
        mask = (all_ev_diffs_short_arr >= bins_short[i]) & (all_ev_diffs_short_arr < bins_short[i + 1])
        bin_outcomes = all_outcomes_short_arr[mask]
        bin_count = len(bin_outcomes)
        if bin_count > 0:
            prop_correct_short.append(np.mean(bin_outcomes))
            bin_counts_short.append(bin_count)
        else:
            prop_correct_short.append(np.nan)
            bin_counts_short.append(0)
    
    prop_correct_short = np.array(prop_correct_short)
    valid_short = ~np.isnan(prop_correct_short)
    
    # Scale sizes: min 30, max 300 based on trial counts
    sizes_short = np.array(bin_counts_short)[valid_short]
    sizes_short = 30 + (sizes_short / sizes_short.max() * 270)  # 30-300 range
    
    plt.scatter(
        bin_centers_short[valid_short],
        prop_correct_short[valid_short],
        s=sizes_short,  # Size varies by trials per bin
        c='green',
        alpha=0.8,
        edgecolors='black',
        linewidth=1.8,
    )
    plt.xlabel('Yellow - Blue Evidence Difference')
    plt.ylabel('P(Yellow is Correct)')
    plt.title(f'Psychometric Curve - SHORT Horizon\n(All Subjects, N={len(all_ev_diffs_short)} trials)')
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.2, 1.2)
    plt.tight_layout()
    plt.show()

# ---------- Summary stats ----------
print(
    f"Long horizon: {len(all_ev_diffs_long)} total trials "
    f"({sum(1 for x in all_outcomes_long if x == 1)} yellow-correct, "
    f"{sum(1 for x in all_outcomes_long if x == 0)} blue-correct, "
    f"{sum(1 for x in all_outcomes_long if x == 0.5)} missing)"
)
print(f"  Mean P(yellow correct): {np.mean(all_outcomes_long):.3f}")

if len(all_ev_diffs_short) > 0:
    print(
        f"Short horizon: {len(all_ev_diffs_short)} total trials "
        f"({sum(1 for x in all_outcomes_short if x == 1)} yellow-correct, "
        f"{sum(1 for x in all_outcomes_short if x == 0)} blue-correct, "
        f"{sum(1 for x in all_outcomes_short if x == 0.5)} missing)"
    )
    print(f"  Mean P(yellow correct): {np.mean(all_outcomes_short):.3f}")


In [ ]:
def plot_psychometric_curve_combined(subject_id, include_missing=True, n_bins=10):
    """
    Plot combined psychometric curve for specific subject (LONG + SHORT)
    P(Yellow is Correct) - both horizons in same bins
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Get subject data for both horizons
    long_data = all_subject_evidence_dicts_of_dict['long'][subject_id]
    short_data = all_subject_evidence_dicts_of_dict['short'][subject_id] if subject_id in all_subject_evidence_dicts_of_dict['short'] else None
    
    ev_diffs = []
    outcomes = []
    
    # Process LONG horizon
    for index, row in long_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        
        # Define whether yellow is actually the correct color
        if yellow > blue:
            yellow_correct = 1.0
        elif yellow < blue:
            yellow_correct = 0.0
        else:
            yellow_correct = 0.5  # tie handling
        
        if outcome == -1 and include_missing:
            ev_diffs.append(ev_diff)
            outcomes.append(0.5)  # fixed from 0.25
        elif outcome != -1:
            ev_diffs.append(ev_diff)
            outcomes.append(yellow_correct)
    
    # Process SHORT horizon (if exists)
    if short_data is not None:
        for index, row in short_data.iterrows():
            last_trial = row['draw_yellow_blue_action_outcome'][-1]
            draw, yellow, blue, action, outcome = last_trial
            
            ev_diff = yellow - blue
            
            if yellow > blue:
                yellow_correct = 1.0
            elif yellow < blue:
                yellow_correct = 0.0
            else:
                yellow_correct = 0.5
            
            if outcome == -1 and include_missing:
                ev_diffs.append(ev_diff)
                outcomes.append(0.5)
            elif outcome != -1:
                ev_diffs.append(ev_diff)
                outcomes.append(yellow_correct)
    
    ev_diffs = np.array(ev_diffs)
    outcomes = np.array(outcomes)
    
    # Combined binning across ALL trials
    bins = np.linspace(ev_diffs.min(), ev_diffs.max(), n_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    prop_correct = []
    bin_counts = []
    for i in range(len(bins)-1):
        mask = (ev_diffs >= bins[i]) & (ev_diffs < bins[i+1])
        bin_outcomes = outcomes[mask]
        bin_count = len(bin_outcomes)
        prop_correct.append(np.mean(bin_outcomes) if bin_count > 0 else np.nan)
        bin_counts.append(bin_count)
    
    # Plotting
    valid = ~np.isnan(prop_correct)
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(bin_centers[valid], np.array(prop_correct)[valid], 
                         s=np.array(bin_counts)[valid]*3,  # size by trial count
                         c='green', alpha=0.8, edgecolors='black', linewidth=1.5)
    plt.xlabel('Yellow - Blue Evidence Difference')
    plt.ylabel('P(Yellow is Correct)')
    plt.title(f'Psychometric Curve - Subject {subject_id} (LONG+SHORT COMBINED)\nN={len(ev_diffs)} total trials')
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.2, 1.2)
    # plt.colorbar(scatter, label='Trials per bin')
    plt.tight_layout()
    plt.show()
    
    # Summary
    n_yellow_correct = sum(outcomes == 1.0)
    n_blue_correct = sum(outcomes == 0.0)
    n_missing = sum(outcomes == 0.5)
    print(f"Subject {subject_id} (COMBINED LONG+SHORT): {len(ev_diffs)} trials")
    print(f"  Yellow-correct: {n_yellow_correct}, Blue-correct: {n_blue_correct}, Missing: {n_missing}")
    print(f"  Mean P(yellow correct): {np.mean(outcomes):.3f}")
    print(f"  Bins with data: {sum(np.array(bin_counts) > 0)}/{n_bins-1}")

# Usage
subject_id = 1
plot_psychometric_curve_combined(subject_id,n_bins=25)


In [ ]:
def plot_psychometric_curve(subject_id, horizon='long', include_missing=True, n_bins=10):
    """
    Plot psychometric curve for specific subject - P(Yellow is Correct)
    Dots sized by trials per bin
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Get subject data
    subject_data = all_subject_evidence_dicts_of_dict[horizon][subject_id]
    
    # Extract data - MODIFIED: yellow_correct = 1 if yellow > blue
    ev_diffs = []
    outcomes = []
    
    for index, row in subject_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        
        # Define whether yellow is actually the correct color
        if yellow > blue:
            yellow_correct = 1.0
        elif yellow < blue:
            yellow_correct = 0.0
        else:
            yellow_correct = 0.5  # tie handling
        
        # if outcome == -1 and include_missing:
        #     ev_diffs.append(ev_diff)
        #     outcomes.append(0.5)  # fixed from 0.25
        if outcome != -1:  # include all non-missing trials
            ev_diffs.append(ev_diff)
            outcomes.append(yellow_correct)
    
    ev_diffs = np.array(ev_diffs)
    outcomes = np.array(outcomes)
    
    # Binning + count trials per bin
    bins = np.linspace(ev_diffs.min(), ev_diffs.max(), n_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    prop_correct = []
    bin_counts = []  # NEW: track trials per bin
    for i in range(len(bins)-1):
        mask = (ev_diffs >= bins[i]) & (ev_diffs < bins[i+1])
        bin_outcomes = outcomes[mask]
        bin_count = len(bin_outcomes)
        prop_correct.append(np.mean(bin_outcomes) if bin_count > 0 else np.nan)
        bin_counts.append(bin_count)
    
    # Plotting - SIZE BY TRIALS
    valid = ~np.isnan(prop_correct)
    valid_bin_counts = np.array(bin_counts)[valid]
    
    # Scale sizes: min 30, max 250 based on trial counts
    sizes = 30 + (valid_bin_counts / valid_bin_counts.max() * 220) if len(valid_bin_counts) > 0 else [120]
    
    plt.figure(figsize=(8, 6))
    plt.scatter(bin_centers[valid], np.array(prop_correct)[valid], 
                s=sizes,  # CHANGED: size varies by trials per bin
                c='green', 
                alpha=0.8, 
                edgecolors='black', 
                linewidth=1.8)
    plt.xlabel('Yellow - Blue Evidence Difference')
    plt.ylabel('P(Yellow is Correct)')
    plt.title(f'Psychometric Curve - Subject {subject_id} ({horizon.upper()})\nN={len(ev_diffs)} trials')
    plt.grid(True, alpha=0.3)
    plt.ylim(-0.2, 1.2)
    plt.tight_layout()
    plt.show()
    
    # Summary
    n_yellow_correct = sum(outcomes == 1.0)
    n_blue_correct = sum(outcomes == 0.0)
    n_missing = sum(outcomes == 0.5)
    print(f"Subject {subject_id} ({horizon.upper()}): {len(ev_diffs)} trials")
    print(f"  Yellow-correct: {n_yellow_correct}, Blue-correct: {n_blue_correct}, Missing: {n_missing}")
    print(f"  Mean P(yellow correct): {np.mean(outcomes):.3f}")

# Usage
subject_id = 50  # define subject_id first
n_bins=30
plot_psychometric_curve(subject_id, 'long',n_bins=n_bins)
plot_psychometric_curve(subject_id, 'short',n_bins=n_bins)


In [ ]:
import sys
import os

# Add project root to path (one level up from notebooks directory)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Remove if already exists
if project_root in sys.path:
    sys.path.remove(project_root)

# Add at the beginning
sys.path.insert(0, project_root)
from src.config import *
print("Project root:", project_root)
print("Python path after update:", sys.path[:3])

In [ ]:
algorithm = ALGORITHM
max_cards_per_draw = MAX_CARDS_PER_DRAW
param_order = PARAM_ORDER
param_ranges = PARAM_RANGES
n_subjects = N_SUBJECTS
n_jobs = N_JOBS
figure_path = FIGURE_PATH
data_path = DATA_PATH
file_path = HUMAN_DATA_PATH
all_evidence_path = ALL_EVIDENCE_PATH
full_sim_df_path = FULL_SIM_DF_PATH
results_path = RESULTS_PATH
full_sim_df_recovered_path = FULL_SIM_DF_RECOVERED_PATH
results_recovered_path = RESULTS_RECOVERED_PATH
is_hazardous = IS_HAZARDOUS

In [ ]:
horizon_condition=FIT_HORIZON[1]

In [ ]:
horizon_condition

In [ ]:

file_path = os.path.join("../data/TrHu_NHB_light/data_MEG/behdat_preprocessed.pkl")
human_data=pd.read_pickle(HUMAN_DATA_PATH)
# extract the subjects. 
human_data_filtered=human_data[:n_subjects]
results_df=pd.read_pickle(results_path)

all_simulated_data = pd.read_pickle(FULL_SIM_DF_PATH_compressed)
results_df_recovered=pd.read_pickle(results_recovered_path)
simulated_subject_dfs_recovered = pd.read_pickle(full_sim_df_recovered_path)


In [ ]:
# git the parameters of the subject_id 
results_df['fit_params_ga'][subject_id]
# print the parameters in a good order according to the param_order, the params is only the list without keys but the same order as the param_order.
params = results_df['fit_params_ga'][subject_id]
for i,param_name in enumerate(param_order):
    print(f"{param_name}: {params[i]}")
    
# plot the softmax given the xi and tau from the fitted parameters:
# xi = params[param_order.index('xi')]
# tau = params[param_order.index('tau')]

def softmax_policy( action_values: np.ndarray, xi: float, tau: float, axis: int = -1) -> np.ndarray:
        """
        Numerically stable softmax with lapse applied uniformly to all actions.
        """
        # Standard, stable softmax
        max_val = np.max(action_values, axis=axis, keepdims=True)
        all_impossible_mask = np.isneginf(max_val)
        safe_max_val = np.where(all_impossible_mask, 0, max_val)
        
        # Calculate scaled values using temperature (tau)
        
        scaled_values = (action_values - safe_max_val) / tau

        exps = np.exp(scaled_values)
        denominator = np.sum(exps, axis=axis, keepdims=True)
        softmax_probs = np.nan_to_num(exps / denominator)

        # Count the total number of actions along the specified axis
        n_actions = action_values.shape[axis]
        final_policy = (1 - xi) * softmax_probs + (xi / n_actions)

        return final_policy
    


In [ ]:
def plot_psychometric_curve(subject_id,all_subject_evidence_dicts_of_dict, horizon='long', n_bins=10, show_lines=True):
    """
    FIXED: Works for both long/short horizons
    """
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Get subject data
    subject_data = all_subject_evidence_dicts_of_dict[horizon][subject_id]
    
    ev_diffs_all = []
    yellow_correct_all = []
    is_missing_all = []
    
    for index, row in subject_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        
        if yellow > blue:
            yellow_correct = 1.0
        elif yellow < blue:
            yellow_correct = 0.0
        else:
            yellow_correct = 0.5
        
        ev_diffs_all.append(ev_diff)
        yellow_correct_all.append(yellow_correct)
        is_missing_all.append(1 if outcome == -1 else 0)
    
    ev_diffs_all = np.array(ev_diffs_all)
    yellow_correct_all = np.array(yellow_correct_all)
    is_missing_all = np.array(is_missing_all)
    
    mask_non_missing = (is_missing_all == 0)
    
    # Binning
    bins = np.linspace(ev_diffs_all.min(), ev_diffs_all.max(), n_bins)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    prop_correct = []
    prop_missing = []
    bin_counts_total = []
    
    for i in range(len(bins)-1):
        mask_all = (ev_diffs_all >= bins[i]) & (ev_diffs_all < bins[i+1])
        bin_counts_all = np.sum(mask_all)
        bin_counts_total.append(bin_counts_all)
        
        if bin_counts_all > 0:
            prop_missing.append(is_missing_all[mask_all].mean())
        else:
            prop_missing.append(np.nan)
        
        mask_nm = mask_non_missing & mask_all
        if np.any(mask_nm):
            nm_indices = np.where(mask_nm)[0]
            bin_outcomes = yellow_correct_all[nm_indices]
            prop_correct.append(np.mean(bin_outcomes))
        else:
            prop_correct.append(np.nan)
    
    # FIXED size calculation function
    def get_sizes(valid_counts):
        if len(valid_counts) == 0:
            return [120]
        return 30 + (valid_counts / valid_counts.max() * 220)
    
    # Plot with DUAL Y-AXES + OPTIONAL LINES
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    # LEFT AXIS: P(Yellow correct) - GREEN
    valid_correct = ~np.isnan(prop_correct)
    valid_centers_correct = bin_centers[valid_correct]
    valid_props_correct = np.array(prop_correct)[valid_correct]
    valid_counts_correct = np.array(bin_counts_total)[valid_correct]
    sizes_correct = get_sizes(valid_counts_correct)  # FIXED
    
    # GREEN dots
    ax1.scatter(valid_centers_correct, valid_props_correct,
                s=sizes_correct, c='green', alpha=0.8, edgecolors='black', linewidth=1.8,
                label='P(Yellow correct | not missing)')
    
    # OPTIONAL GREEN line
    if show_lines and len(valid_centers_correct) > 1:
        ax1.plot(valid_centers_correct, valid_props_correct, 'g-', alpha=0.8, linewidth=2)
    
    ax1.set_xlabel('Yellow - Blue Evidence Difference')
    ax1.set_ylabel('P(Yellow correct | not missing)', color='green')
    ax1.tick_params(axis='y', labelcolor='green')
    ax1.grid(True, alpha=0.3)
    aligned_ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    ax1.set_yticks(aligned_ticks)
    ax1.set_ylim(-0.2, 1.2)
    
    # RIGHT AXIS: P(Miss) - RED
    ax2 = ax1.twinx()
    valid_missing = ~np.isnan(prop_missing)
    valid_centers_missing = bin_centers[valid_missing]
    valid_props_missing = np.array(prop_missing)[valid_missing]
    valid_counts_missing = np.array(bin_counts_total)[valid_missing]
    sizes_missing = get_sizes(valid_counts_missing)  # FIXED
    
    # RED dots
    ax2.scatter(valid_centers_missing, valid_props_missing,
                s=sizes_missing, c='red', alpha=0.8, edgecolors='darkred', linewidth=1.8,
                label='P(Miss)')
    
    # OPTIONAL RED line
    if show_lines and len(valid_centers_missing) > 1:
        ax2.plot(valid_centers_missing, valid_props_missing, 'r-', alpha=0.8, linewidth=2)
    
    ax2.set_ylabel('P(Miss)', color='red')
    ax2.tick_params(axis='y', labelcolor='red')
    ax2.set_yticks(aligned_ticks)
    ax2.set_ylim(-0.2, 1.2)
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', bbox_to_anchor=(0, 1.02))
    
    plt.title(f'Psychometric Curves - Subject {subject_id} ({horizon.upper()})\n'
              f'N={len(ev_diffs_all)} trials')
    
    plt.tight_layout()
    plt.show()
    
    # Summary
    outcomes = yellow_correct_all[mask_non_missing]
    n_yellow_correct = sum(outcomes == 1.0)
    n_blue_correct = sum(outcomes == 0.0)
    n_missing = sum(is_missing_all == 1)
    print(f"Subject {subject_id} ({horizon.upper()}): {len(ev_diffs_all)} trials")
    print(f"  Yellow-correct: {n_yellow_correct}, Blue-correct: {n_blue_correct}, Missing: {n_missing}")
    print(f"  Mean P(yellow correct | non-missing): {np.mean(outcomes):.3f}")
    print(f"  Mean P(miss): {np.mean(is_missing_all):.3f}")



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_psychometric_comparison(human_data_dict, simulated_data_dict, subject_id, horizon='short', n_bins=12, show_lines=True):
    """
    Compare human vs simulated psychometric curves with per-bin errors
    """
    # Get human data
    human_subject_data = human_data_dict[horizon][subject_id]
    ev_diffs_human, yellow_correct_human, is_missing_human = extract_evidence_outcomes(human_subject_data)
    
    # Get simulated data  
    sim_subject_data = simulated_data_dict[horizon][subject_id]
    ev_diffs_sim, yellow_correct_sim, is_missing_sim = extract_evidence_outcomes(sim_subject_data)
    
    # Bin both datasets using same bins
    bins, bin_centers = create_bins(ev_diffs_human, n_bins)
    
    # Compute binned stats for both
    human_stats = compute_bin_stats(ev_diffs_human, yellow_correct_human, is_missing_human, bins)
    sim_stats = compute_bin_stats(ev_diffs_sim, yellow_correct_sim, is_missing_sim, bins)
    
    # Calculate errors
    mse_correct = np.nanmean((human_stats['prop_correct'] - sim_stats['prop_correct'])**2)
    mae_correct = np.nanmean(np.abs(human_stats['prop_correct'] - sim_stats['prop_correct']))
    
    # Plot comparison
    plot_comparison(bin_centers, human_stats, sim_stats, subject_id, horizon, mse_correct, mae_correct)
    
    print_summary(subject_id, horizon, human_stats, sim_stats, mse_correct, mae_correct)

def extract_evidence_outcomes(subject_data):
    """Extract EV diffs, yellow_correct, and missing from subject data"""
    ev_diffs_all = []
    yellow_correct_all = []
    is_missing_all = []
    
    for index, row in subject_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial
        
        ev_diff = yellow - blue
        yellow_correct = 1.0 if yellow > blue else (0.0 if yellow < blue else 0.5)
        is_missing = 1 if outcome == -1 else 0
        
        ev_diffs_all.append(ev_diff)
        yellow_correct_all.append(yellow_correct)
        is_missing_all.append(is_missing)
    
    return np.array(ev_diffs_all), np.array(yellow_correct_all), np.array(is_missing_all)

def create_bins(ev_diffs, n_bins):
    """Create bins and bin centers"""
    bins = np.linspace(ev_diffs.min(), ev_diffs.max(), n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    return bins, bin_centers

def compute_bin_stats(ev_diffs, yellow_correct, is_missing, bins):
    """Compute proportion correct and missing per bin"""
    mask_non_missing = (is_missing == 0)
    
    prop_correct = []
    prop_missing = []
    bin_counts = []
    
    for i in range(len(bins)-1):
        mask_bin = (ev_diffs >= bins[i]) & (ev_diffs < bins[i+1])
        bin_count = np.sum(mask_bin)
        bin_counts.append(bin_count)
        
        # P(missing)
        if bin_count > 0:
            prop_missing.append(is_missing[mask_bin].mean())
        else:
            prop_missing.append(np.nan)
        
        # P(correct | not missing)
        mask_nm_bin = mask_non_missing & mask_bin
        if np.any(mask_nm_bin):
            prop_correct.append(yellow_correct[mask_nm_bin].mean())
        else:
            prop_correct.append(np.nan)
    
    return {
        'prop_correct': np.array(prop_correct),
        'prop_missing': np.array(prop_missing), 
        'bin_counts': np.array(bin_counts),
        'bin_centers': (bins[:-1] + bins[1:]) / 2
    }

def get_point_sizes(valid_counts):
    """Size points proportional to bin counts"""
    if len(valid_counts) == 0:
        return [120]
    return 30 + (valid_counts / valid_counts.max() * 220)

def plot_comparison(bin_centers, human_stats, sim_stats, subject_id, horizon, mse_correct, mae_correct):
    """Create dual-axis comparison plot"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Left: P(correct | not missing) comparison
    plot_metric_comparison(ax1, bin_centers, human_stats['prop_correct'], sim_stats['prop_correct'], 
                          human_stats['bin_counts'], sim_stats['bin_counts'],
                          'P(Yellow correct | not missing)', 'green', show_lines=True)
    ax1.legend(['Human', 'Simulated', 'Human fit', 'Sim fit'])
    
    # Right: P(missing) comparison  
    plot_metric_comparison(ax2, bin_centers, human_stats['prop_missing'], sim_stats['prop_missing'],
                          human_stats['bin_counts'], sim_stats['bin_counts'],
                          'P(Miss)', 'red', show_lines=True)
    ax2.legend(['Human', 'Simulated', 'Human fit', 'Sim fit'])
    
    plt.suptitle(f'Psychometric Curves Comparison - Subject {subject_id} ({horizon.upper()})\n'
                f'MSE(correct)={mse_correct:.3f}, MAE(correct)={mae_correct:.3f}', fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_metric_comparison(ax, bin_centers, human_vals, sim_vals, human_counts, sim_counts, ylabel, color, show_lines):
    """Plot single metric comparison"""
    # Human data
    valid_human = ~np.isnan(human_vals)
    ax.scatter(bin_centers[valid_human], human_vals[valid_human],
              s=get_point_sizes(human_counts[valid_human]), c=color, alpha=0.8,
              edgecolors='black', linewidth=1.5, label='Human', zorder=3)
    
    # Simulated data  
    valid_sim = ~np.isnan(sim_vals)
    ax.scatter(bin_centers[valid_sim], sim_vals[valid_sim],
              s=get_point_sizes(sim_counts[valid_sim]), facecolors='none', 
              edgecolors=color, linewidth=2.5, label='Simulated', zorder=4)
    
    # Lines
    if show_lines:
        ax.plot(bin_centers[valid_human], human_vals[valid_human], color=color, alpha=0.7, linewidth=2, label='Human fit')
        ax.plot(bin_centers[valid_sim], sim_vals[valid_sim], color=color, linestyle='--', alpha=0.7, linewidth=2, label='Sim fit')
    
    ax.set_xlabel('Yellow - Blue Evidence Difference')
    ax.set_ylabel(ylabel, color=color)
    ax.tick_params(axis='y', labelcolor=color)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.1, 1.1)
    ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])

def print_summary(subject_id, horizon, human_stats, sim_stats, mse_correct, mae_correct):
    """Print comparison summary"""
    human_correct = human_stats['prop_correct'][~np.isnan(human_stats['prop_correct'])]
    sim_correct = sim_stats['prop_correct'][~np.isnan(sim_stats['prop_correct'])]
    
    print(f"\nSubject {subject_id} ({horizon.upper()}) COMPARISON:")
    print(f"  MSE (P(correct|not missing)): {mse_correct:.3f}")
    print(f"  MAE (P(correct|not missing)): {mae_correct:.3f}")
    print(f"  Human mean P(correct|not missing): {np.nanmean(human_correct):.3f}")
    print(f"  Sim   mean P(correct|not missing): {np.nanmean(sim_correct):.3f}")
    print(f"  Human mean P(missing): {np.nanmean(human_stats['prop_missing']):.3f}")
    print(f"  Sim   mean P(missing): {np.nanmean(sim_stats['prop_missing']):.3f}")

# Usage:
subject_id = 1
n_bins = 12


# Compare human vs simulated for this subject/horizon
plot_psychometric_comparison(all_subject_evidence_dicts_of_dict, all_simulated_data, 
                           subject_id, horizon=horizon_condition, n_bins=n_bins)


In [ ]:
all_simulated_data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. EXTRACT TRIAL-LEVEL VARIABLES
# ==========================================

def extract_evidence_outcomes(subject_data):
    ev_diffs_all = []
    yellow_correct_all = []
    is_missing_all = []

    for _, row in subject_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial

        ev_diff = yellow - blue
        yellow_correct = 1.0 if yellow > blue else (0.0 if yellow < blue else 0.5)
        is_missing = 1 if outcome == -1 else 0

        ev_diffs_all.append(ev_diff)
        yellow_correct_all.append(yellow_correct)
        is_missing_all.append(is_missing)

    return (
        np.array(ev_diffs_all),
        np.array(yellow_correct_all),
        np.array(is_missing_all)
    )

# ==========================================
# 2. BIN STATISTICS
# ==========================================

def compute_bin_stats_detailed(ev_diffs, yellow_correct, is_missing, bins):

    mask_non_missing = (is_missing == 0)

    prop_correct = []
    prop_missing = []
    n_correct = []
    n_total_correct = []
    n_missing = []
    n_total_missing = []

    for i in range(len(bins) - 1):

        mask_bin = (ev_diffs >= bins[i]) & (ev_diffs < bins[i + 1])
        n_bin = np.sum(mask_bin)

        # P(missing)
        n_miss_bin = np.sum(is_missing[mask_bin])
        prop_missing.append(n_miss_bin / n_bin if n_bin > 0 else np.nan)
        n_missing.append(n_miss_bin)
        n_total_missing.append(n_bin)

        # P(correct | not missing)
        mask_nm_bin = mask_non_missing & mask_bin
        n_nm_bin = np.sum(mask_nm_bin)

        if n_nm_bin > 0:
            n_correct_bin = np.sum(yellow_correct[mask_nm_bin] == 1.0)
            prop_correct.append(n_correct_bin / n_nm_bin)
            n_correct.append(n_correct_bin)
            n_total_correct.append(n_nm_bin)
        else:
            prop_correct.append(np.nan)
            n_correct.append(0)
            n_total_correct.append(0)

    return {
        "prop_correct": np.array(prop_correct),
        "prop_missing": np.array(prop_missing),
        "n_correct": np.array(n_correct),
        "n_total_correct": np.array(n_total_correct),
        "n_missing": np.array(n_missing),
        "n_total_missing": np.array(n_total_missing),
    }

# ==========================================
# 3. GLOBAL BINS ACROSS ALL SUBJECTS
# ==========================================

def compute_global_bins(human_data_dict, simulated_data_dict, horizon, n_bins):

    all_ev = []

    for subject_id in human_data_dict[horizon].keys():
        ev_h, _, _ = extract_evidence_outcomes(
            human_data_dict[horizon][subject_id]
        )
        ev_s, _, _ = extract_evidence_outcomes(
            simulated_data_dict[horizon][subject_id]
        )

        all_ev.append(ev_h)
        all_ev.append(ev_s)

    all_ev = np.concatenate(all_ev)

    bins = np.linspace(all_ev.min(), all_ev.max(), n_bins + 1)
    centers = (bins[:-1] + bins[1:]) / 2

    return bins, centers

# ==========================================
# 4. ALL-SUBJECT DIFFERENCES
# ==========================================

def compute_all_subject_differences(human_data_dict,
                                    simulated_data_dict,
                                    horizon='short',
                                    n_bins=20):

    bins, centers = compute_global_bins(
        human_data_dict,
        simulated_data_dict,
        horizon,
        n_bins
    )

    subject_ids = human_data_dict[horizon].keys()

    all_correct_diffs = []
    all_missing_diffs = []

    for subject_id in subject_ids:

        ev_h, yc_h, miss_h = extract_evidence_outcomes(
            human_data_dict[horizon][subject_id]
        )
        ev_s, yc_s, miss_s = extract_evidence_outcomes(
            simulated_data_dict[horizon][subject_id]
        )

        human_stats = compute_bin_stats_detailed(ev_h, yc_h, miss_h, bins)
        sim_stats   = compute_bin_stats_detailed(ev_s, yc_s, miss_s, bins)

        diff_correct = human_stats["prop_correct"] - sim_stats["prop_correct"]
        diff_missing = human_stats["prop_missing"] - sim_stats["prop_missing"]

        all_correct_diffs.append(diff_correct)
        all_missing_diffs.append(diff_missing)

    return (
        np.array(all_correct_diffs),
        np.array(all_missing_diffs),
        centers
    )

# ==========================================
# 5. ROBUST GROUP AGGREGATION (NO WARNINGS)
# ==========================================

def aggregate_group(all_diffs):

    valid_counts = np.sum(~np.isnan(all_diffs), axis=0)

    mean = np.where(
        valid_counts > 0,
        np.nanmean(all_diffs, axis=0),
        np.nan
    )

    sem = np.where(
        valid_counts > 1,
        np.nanstd(all_diffs, axis=0) / np.sqrt(valid_counts),
        np.nan
    )

    return mean, sem

# ==========================================
# 6. GLOBAL FIT METRICS
# ==========================================

def compute_global_metrics(all_diffs):

    return {
        "MSE": float(np.nanmean(all_diffs ** 2)),
        "MAE": float(np.nanmean(np.abs(all_diffs))),
        "Mean Bias": float(np.nanmean(all_diffs))
    }

# ==========================================
# 7. GROUP PLOTTING
# ==========================================

def plot_group_results(centers,
                       mean_correct,
                       sem_correct,
                       mean_missing,
                       sem_missing):

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Correct
    ax1.axhline(0, color='black', linewidth=1)
    ax1.errorbar(centers, mean_correct, yerr=sem_correct,
                 fmt='o', capsize=4)
    ax1.set_title("Group: Human - Sim (P correct | not missing)")
    ax1.set_xlabel("Evidence Difference")
    ax1.set_ylabel("Difference")
    ax1.set_ylim(-0.4, 0.4)
    ax1.grid(alpha=0.3)

    # Missing
    ax2.axhline(0, color='black', linewidth=1)
    ax2.errorbar(centers, mean_missing, yerr=sem_missing,
                 fmt='o', capsize=4)
    ax2.set_title("Group: Human - Sim (P missing)")
    ax2.set_xlabel("Evidence Difference")
    ax2.set_ylim(-0.4, 0.4)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

# ==========================================
# 8. RUN SYSTEMATIC EVALUATION
# ==========================================

all_correct, all_missing, centers = compute_all_subject_differences(
    all_subject_evidence_dicts_of_dict,
    all_simulated_data,
    horizon=horizon_condition,
    n_bins=20
)

mean_correct, sem_correct = aggregate_group(all_correct)
mean_missing, sem_missing = aggregate_group(all_missing)

print("Correct Fit Metrics:", compute_global_metrics(all_correct))
print("Missing Fit Metrics:", compute_global_metrics(all_missing))

plot_group_results(
    centers,
    mean_correct,
    sem_correct,
    mean_missing,
    sem_missing
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# ==========================================
# 1. EXTRACT TRIAL-LEVEL VARIABLES
# ==========================================

def extract_evidence_outcomes(subject_data):
    ev_diffs_all = []
    yellow_correct_all = []
    is_missing_all = []

    for _, row in subject_data.iterrows():
        last_trial = row['draw_yellow_blue_action_outcome'][-1]
        draw, yellow, blue, action, outcome = last_trial

        ev_diff = yellow - blue
        yellow_correct = 1.0 if yellow > blue else (0.0 if yellow < blue else 0.5)
        is_missing = 1 if outcome == -1 else 0

        ev_diffs_all.append(ev_diff)
        yellow_correct_all.append(yellow_correct)
        is_missing_all.append(is_missing)

    return (
        np.array(ev_diffs_all),
        np.array(yellow_correct_all),
        np.array(is_missing_all)
    )

# ==========================================
# 2. BIN STATISTICS
# ==========================================

def compute_bin_stats(ev_diffs, yellow_correct, is_missing, bins):

    mask_non_missing = (is_missing == 0)

    prop_correct = []
    prop_missing = []

    for i in range(len(bins) - 1):

        mask_bin = (ev_diffs >= bins[i]) & (ev_diffs < bins[i + 1])
        n_bin = np.sum(mask_bin)

        # Missing
        if n_bin > 0:
            prop_missing.append(np.sum(is_missing[mask_bin]) / n_bin)
        else:
            prop_missing.append(np.nan)

        # Correct | not missing
        mask_nm_bin = mask_non_missing & mask_bin
        n_nm_bin = np.sum(mask_nm_bin)

        if n_nm_bin > 0:
            prop_correct.append(
                np.sum(yellow_correct[mask_nm_bin] == 1.0) / n_nm_bin
            )
        else:
            prop_correct.append(np.nan)

    return np.array(prop_correct), np.array(prop_missing)

# ==========================================
# 3. GLOBAL BINS
# ==========================================

def compute_global_bins(human_data_dict, simulated_data_dict, horizon, n_bins):

    all_ev = []

    for subject_id in human_data_dict[horizon].keys():
        ev_h, _, _ = extract_evidence_outcomes(
            human_data_dict[horizon][subject_id]
        )
        ev_s, _, _ = extract_evidence_outcomes(
            simulated_data_dict[horizon][subject_id]
        )

        all_ev.append(ev_h)
        all_ev.append(ev_s)

    all_ev = np.concatenate(all_ev)

    bins = np.linspace(all_ev.min(), all_ev.max(), n_bins + 1)
    centers = (bins[:-1] + bins[1:]) / 2

    return bins, centers

# ==========================================
# 4. COMPREHENSIVE SUBJECT PLOT WITH RELATIVE DOT SIZES
# ==========================================
def plot_all_subjects_psychometric_difference(human_data_dict,
                                              simulated_data_dict,
                                              horizon='short',
                                              n_bins=20):

    bins, centers = compute_global_bins(
        human_data_dict,
        simulated_data_dict,
        horizon,
        n_bins
    )

    subject_ids = list(human_data_dict[horizon].keys())
    n_subjects = len(subject_ids)

    n_cols = 5
    n_rows = math.ceil(n_subjects / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5*n_cols, 3*n_rows),
                             sharex=True,
                             sharey=True)

    axes = axes.flatten()

    for idx, subject_id in enumerate(subject_ids):

        ax = axes[idx]

        # Extract
        ev_h, yc_h, miss_h = extract_evidence_outcomes(
            human_data_dict[horizon][subject_id]
        )
        ev_s, yc_s, miss_s = extract_evidence_outcomes(
            simulated_data_dict[horizon][subject_id]
        )

        # Compute psychometric curves
        human_correct, human_missing = compute_bin_stats(
            ev_h, yc_h, miss_h, bins
        )
        sim_correct, sim_missing = compute_bin_stats(
            ev_s, yc_s, miss_s, bins
        )

        # Compute trial counts SEPARATELY for each curve
        n_choice_h = []
        n_choice_s = []
        n_missing_h = []
        n_missing_s = []
        
        for i in range(len(bins) - 1):
            mask_bin_h = (ev_h >= bins[i]) & (ev_h < bins[i + 1])
            mask_bin_s = (ev_s >= bins[i]) & (ev_s < bins[i + 1])
            
            # Choice trials (non-missing)
            mask_choice_h = mask_bin_h & (miss_h == 0)
            mask_choice_s = mask_bin_s & (miss_s == 0)
            n_choice_h.append(np.sum(mask_choice_h))
            n_choice_s.append(np.sum(mask_choice_s))
            
            # Missing trials
            mask_missing_h = mask_bin_h & (miss_h == 1)
            mask_missing_s = mask_bin_s & (miss_s == 1)
            n_missing_h.append(np.sum(mask_missing_h))
            n_missing_s.append(np.sum(mask_missing_s))
        
        # Filter valid bins only
        valid_mask = ~np.isnan(human_correct)
        centers_valid = centers[valid_mask]
        
        n_choice = np.array(n_choice_h)[valid_mask] + np.array(n_choice_s)[valid_mask]
        n_missing = np.array(n_missing_h)[valid_mask] + np.array(n_missing_s)[valid_mask]
        
        # RELATIVE SIZING: scale each to its own max (30-250 range)
        sizes_choice = 30 + (n_choice / n_choice.max() * 220) if len(n_choice) > 0 and n_choice.max() > 0 else np.full_like(n_choice, 50)
        sizes_missing = 30 + (n_missing / n_missing.max() * 220) if len(n_missing) > 0 and n_missing.max() > 0 else np.full_like(n_missing, 50)

        # Differences (valid only)
        delta_correct = human_correct[valid_mask] - sim_correct[valid_mask]
        delta_missing = human_missing[valid_mask] - sim_missing[valid_mask]

        # Zero reference
        ax.axhline(0, color='black', linewidth=1)

        # Choice diff (sized by CHOICE trials only)
        ax.scatter(centers_valid, delta_correct, s=sizes_choice, zorder=5,
                   label='Δ P(Choose Yellow | Not Missing)',
                   facecolors='none', edgecolors='C0', linewidth=1.5)
        ax.plot(centers_valid, delta_correct, 'o-', markersize=0, alpha=0.85, zorder=4)

        # Missing diff (sized by MISSING trials only - RELATIVE scale)
        ax.scatter(centers_valid, delta_missing, s=sizes_missing, marker='s', zorder=5,
                   label='Δ P(Missing)',
                   facecolors='none', edgecolors='C1', linewidth=1.5)
        ax.plot(centers_valid, delta_missing, 's--', markersize=0, alpha=0.85, zorder=4)

        ax.set_title(f'Subject {subject_id}', fontsize=9)
        ax.set_ylim(-1, 1)
        ax.grid(alpha=0.25)

    # Remove unused panels
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    # Global labels
    fig.supxlabel("Evidence Difference (Yellow − Blue)")
    fig.supylabel(
        "Δ Psychometric Function (Human − Model)\n"
        "Positive = Humans More Likely"
    )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels,
               loc='center left',
               bbox_to_anchor=(1.02, 0.9),
               borderaxespad=0.5,
               fontsize=9)

    plt.tight_layout()
    plt.show()


# ==========================================
# 5. RUN
# ==========================================

plot_all_subjects_psychometric_difference(
    all_subject_evidence_dicts_of_dict,
    all_simulated_data,
    horizon=horizon_condition,
    n_bins=20
)


## Majority stability: decision-time vs. full-sequence-end

For real decision trials (excludes missed/timeout trials, where the last
recorded action is "wait"), compare the yellow/blue majority at the draw
where the subject actually decided against the majority at the end of the
full underlying card sequence (which continues a few draws past the
decision -- `data_dict_of_lists_fullsequence` records it, `data_dict_of_lists`
truncates it at the decision).

This tests whether an *early* decision is looking at meaningfully less
reliable evidence than a *late* one: if the majority frequently flips
between the decision point and the full-sequence end for early decisions
but rarely for late ones, that's the human data itself confirming that
early evidence is noisier -- which is the reference point for judging
whether a model that decides too early (see the vanilla-model discussion)
*should* be showing degraded outcome accuracy, and by how much.

In [ ]:
import pandas as pd

# data_dict_of_lists / data_dict_of_lists_fullsequence are the subject's raw
# behavioral sequences (independent of which POMDP was fit to them) -- reusing
# them from an existing results.pkl avoids re-deriving the same per-trial
# parsing already done for the fitting pipeline. Use a combined-horizon config
# so both short and long are available from one file.
majority_results_path = os.path.join(project_root, "data/POMDP/CBEXT-RP-CL--/ga/results.pkl")
majority_results_df = pd.read_pickle(majority_results_path)

def majority_label(cum_yellow, cum_blue):
    if cum_yellow > cum_blue:
        return "yellow"
    if cum_blue > cum_yellow:
        return "blue"
    return "tie"

rows = []
for _, subj_row in majority_results_df.iterrows():
    sid = subj_row["subject_ID"]
    for horizon, decided_df in subj_row["data_dict_of_lists"].items():
        full_df = subj_row["data_dict_of_lists_fullsequence"][horizon]
        for i in range(len(decided_df)):
            decided_seq = decided_df.iloc[i]["draw_yellow_blue_action_outcome"]
            full_seq = full_df.iloc[i]["draw_yellow_blue_action_outcome"]

            last_decided = decided_seq[-1]
            draw_idx, cum_yellow_dec, cum_blue_dec, action, outcome = last_decided
            if action == 2:
                continue  # missed/timeout trial -- excluded per the task

            last_full = full_seq[-1]
            full_draw_idx, cum_yellow_end, cum_blue_end = last_full[0], last_full[1], last_full[2]

            maj_decision = majority_label(cum_yellow_dec, cum_blue_dec)
            maj_end = majority_label(cum_yellow_end, cum_blue_end)

            rows.append({
                "subject_ID": sid,
                "horizon": horizon,
                "trial": i,
                "num_draws_decision": draw_idx,
                "num_draws_full": full_draw_idx,
                "extra_draws": full_draw_idx - draw_idx,
                "majority_decision": maj_decision,
                "majority_end": maj_end,
                "same_majority": maj_decision == maj_end,
                "outcome": outcome,
            })

majority_df = pd.DataFrame(rows)
print(f"{len(majority_df)} decision trials across {majority_df['subject_ID'].nunique()} subjects (missed trials excluded)")
majority_df.head()


In [ ]:
agreement_rate = majority_df["same_majority"].mean()
print(f"Overall: majority at decision matches majority at full-sequence end in {agreement_rate:.1%} of decision trials ({len(majority_df)} trials)")

no_tie = majority_df[(majority_df["majority_decision"] != "tie") & (majority_df["majority_end"] != "tie")]
print(f"Excluding ties on either side: {no_tie['same_majority'].mean():.1%} ({len(no_tie)}/{len(majority_df)} trials)")

print()
print("By horizon:")
print(majority_df.groupby("horizon")["same_majority"].agg(["mean", "count"]))


In [ ]:
# The key breakdown for the vanilla-model diagnostic: does majority stability
# depend on how early the decision was made? If early decisions (few draws)
# show much lower agreement than late ones, a model that decides too early
# should show correspondingly worse outcome accuracy -- if it doesn't, that's
# the anomaly worth chasing as a real bug rather than a model-capacity limit.
by_draws = majority_df.groupby("num_draws_decision")["same_majority"].agg(["mean", "count"])
print(by_draws)


### Same analysis, but using the *first* draw specifically

The breakdown by `num_draws_decision` above mixes "decided early" trials
(few of them) with "decided late" ones. To isolate purely how reliable
*draw-1 alone* is, redo the comparison using every trial's actual first
draw (regardless of when the subject decided) against the same
full-sequence-end majority.

In [ ]:
rows_first = []
for _, subj_row in majority_results_df.iterrows():
    sid = subj_row["subject_ID"]
    for horizon, decided_df in subj_row["data_dict_of_lists"].items():
        full_df = subj_row["data_dict_of_lists_fullsequence"][horizon]
        for i in range(len(decided_df)):
            decided_seq = decided_df.iloc[i]["draw_yellow_blue_action_outcome"]
            full_seq = full_df.iloc[i]["draw_yellow_blue_action_outcome"]
            action = decided_seq[-1][3]
            if action == 2:
                continue  # missed/timeout trial -- excluded per the task

            first_entry = full_seq[0]
            cum_yellow_1, cum_blue_1 = first_entry[1], first_entry[2]
            last_full = full_seq[-1]
            cum_yellow_end, cum_blue_end = last_full[1], last_full[2]

            rows_first.append({
                "subject_ID": sid,
                "horizon": horizon,
                "trial": i,
                "num_draws_full": last_full[0],
                "majority_first_draw": majority_label(cum_yellow_1, cum_blue_1),
                "majority_end": majority_label(cum_yellow_end, cum_blue_end),
            })

first_draw_df = pd.DataFrame(rows_first)
first_draw_df["same_majority"] = first_draw_df["majority_first_draw"] == first_draw_df["majority_end"]

print(f"{len(first_draw_df)} decision trials")
print(f"Overall (draw 1 vs. full-sequence end): {first_draw_df['same_majority'].mean():.1%}")
no_tie_first = first_draw_df[
    (first_draw_df["majority_first_draw"] != "tie") & (first_draw_df["majority_end"] != "tie")
]
print(f"Excluding ties: {no_tie_first['same_majority'].mean():.1%} ({len(no_tie_first)}/{len(first_draw_df)})")
print()
print("By horizon:")
print(first_draw_df.groupby("horizon")["same_majority"].agg(["mean", "count"]))


In [ ]:
# Compare directly: majority-at-actual-decision vs. majority-at-first-draw,
# both against the same full-sequence-end ground truth. The gap between
# these two numbers is how much reliability is gained by waiting for more
# evidence -- the benchmark a model's own draw-1 outcome accuracy should be
# judged against (not the model's overall/aggregate outcome accuracy).
print(f"Majority at actual decision matches end: {majority_df['same_majority'].mean():.1%}")
print(f"Majority at draw 1 only matches end:     {first_draw_df['same_majority'].mean():.1%}")


### Benchmark: does the simulated model's accuracy-by-num_draws curve match this?

The real test of whether a model deciding "too early" should show degraded
outcome accuracy: load its raw ensemble simulations and compute accuracy
conditioned on `num_draws`, then compare against the human draw-1-vs-end /
decision-vs-end baselines above. If the simulated per-draw-count accuracy
is comparable to the human numbers (not suspiciously higher), the model's
outcome computation is behaving correctly -- the "outcome distribution
looks fine despite a bad draws fit" is then a property of this task (even
early evidence isn't *that* unreliable here) rather than a code bug.

In [ ]:
import glob

def simulated_accuracy_by_draws(task, horizon, commit=True, n_runs=60):
    subdir = "POMDP_commit" if commit else "POMDP"
    pattern = os.path.join(
        project_root, f"data/{subdir}/{task}/ga/{horizon}/raw_simulations/sim_run_*_{horizon}.pkl"
    )
    files = sorted(glob.glob(pattern))[:n_runs]
    sim = pd.concat([pd.read_pickle(f) for f in files], ignore_index=True)
    decided = sim[sim["outcome"] != -1].copy()
    decided["correct"] = decided["outcome"] == 2
    return decided.groupby("num_draws")["correct"].agg(["mean", "count"]), decided["correct"].mean()

by_draws, overall = simulated_accuracy_by_draws("S--XT-R-h----", "short", commit=True)
print("Simulated accuracy by num_draws (S--XT-R-h----, vanilla, commit, short):")
print(by_draws)
print()
print(f"Overall simulated accuracy: {overall:.1%}")
print(f"Simulated draw-1 accuracy:  {by_draws.loc[1, 'mean']:.1%}" if 1 in by_draws.index else "no draw-1 trials")
print()
print(f"Human draw-1-vs-end agreement (short horizon, from above): {first_draw_df[first_draw_df['horizon']=='short']['same_majority'].mean():.1%}")
print(f"Human decision-vs-end agreement (short horizon, from above): {majority_df[majority_df['horizon']=='short']['same_majority'].mean():.1%}")


### Average human accuracy across subjects, vs. the draw-1 ceiling

Report per-subject accuracy first, then average across subjects (each
subject counted once), rather than pooling all trials together -- the more
standard way to summarize this and the fairer comparison against the
draw-1-vs-end ceiling.

In [ ]:
majority_df["correct"] = majority_df["outcome"] == 2

per_subject_accuracy = majority_df.groupby("subject_ID")["correct"].mean()
per_subject_draw1 = first_draw_df.groupby("subject_ID")["same_majority"].mean()

print(f"{majority_df['subject_ID'].nunique()} subjects")
print()
print("Average human accuracy across subjects (per-subject mean, then averaged):")
print(f"  Actual accuracy:       {per_subject_accuracy.mean():.1%}  (SD={per_subject_accuracy.std():.1%} across subjects)")
print(f"  Draw-1-vs-end ceiling: {per_subject_draw1.mean():.1%}  (SD={per_subject_draw1.std():.1%} across subjects)")


### Generalizing draw-1-vs-end to draw-N-vs-end, for N = 1..10

Same idea as the draw-1 ceiling above, but for every cumulative draw count N:
take the yellow/blue majority using only the first N draws of each trial's
full sequence, and check whether it matches the majority once the sequence
actually concludes. Only trials whose full sequence has *more* than N draws
are included at each N (comparing N to itself when N is already the end
would trivially always "match"). Averaged per subject first, then across
subjects, same convention as above.

In [ ]:
# trials_for_draw_n_analysis needs to exist before the cell above -- build it
# here from the same results_df used throughout this section (one row per
# decision trial: subject_ID + its full draw_yellow_blue_action_outcome
# sequence), so the draw-N loop above only has to walk it once per N.
trials_for_draw_n_analysis = []
for _, subj_row in majority_results_df.iterrows():
    sid = subj_row["subject_ID"]
    for horizon, decided_df in subj_row["data_dict_of_lists"].items():
        full_df = subj_row["data_dict_of_lists_fullsequence"][horizon]
        for i in range(len(decided_df)):
            decided_seq = decided_df.iloc[i]["draw_yellow_blue_action_outcome"]
            if decided_seq[-1][3] == 2:
                continue  # missed/timeout trial -- excluded per the task
            full_seq = full_df.iloc[i]["draw_yellow_blue_action_outcome"]
            trials_for_draw_n_analysis.append((sid, full_seq))

print(f"{len(trials_for_draw_n_analysis)} decision trials")


In [ ]:
max_n = 10
records = []
for n in range(1, max_n + 1):
    for sid, full_seq in trials_for_draw_n_analysis:
        if n >= len(full_seq):
            continue  # need at least one more draw after n to define a non-trivial "end"
        entry_n = full_seq[n - 1]
        entry_end = full_seq[-1]
        maj_n = majority_label(entry_n[1], entry_n[2])
        maj_end = majority_label(entry_end[1], entry_end[2])
        records.append({"n": n, "subject_ID": sid, "match": maj_n == maj_end})

draw_n_df = pd.DataFrame(records)
per_subject_by_n = draw_n_df.groupby(["n", "subject_ID"])["match"].mean().reset_index()
draw_n_summary = per_subject_by_n.groupby("n")["match"].agg(["mean", "std", "count"])
draw_n_summary.columns = ["mean_across_subjects", "sd_across_subjects", "n_subjects"]
draw_n_summary["n_trials"] = draw_n_df.groupby("n").size()
draw_n_summary


### Plot: evidence reliability grows with more draws

In [ ]:
from src.utils.plotting import _set_plot_style, _save_figure

_set_plot_style(font_size=16)

fig, ax = plt.subplots(figsize=(8, 5.5))

x = draw_n_summary.index.to_numpy()
mean = draw_n_summary["mean_across_subjects"].to_numpy()
sd = draw_n_summary["sd_across_subjects"].to_numpy()

ax.fill_between(x, mean - sd, mean + sd, color="steelblue", alpha=0.15, linewidth=0)
ax.plot(x, mean, color="steelblue", linewidth=2, marker="o", markersize=8, zorder=3)

actual_accuracy = per_subject_accuracy.mean()
ax.axhline(
    actual_accuracy, color="0.4", linestyle="--", linewidth=1.5, zorder=2,
    label=f"Actual human accuracy ({actual_accuracy:.0%})",
)
ax.axhline(0.5, color="0.75", linestyle=":", linewidth=1, zorder=1, label="Chance (50%)")

ax.set_xlabel("Draws used (cumulative)")
ax.set_ylabel("Agreement with eventual majority")
ax.set_title("How reliable is the evidence after N draws?\n(human card sequences, decision trials only)")
ax.set_xticks(x)
ax.set_ylim(0.4, 1.0)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=True, loc="lower right")
fig.tight_layout()

_save_figure(fig, "draw_n_vs_end_majority_agreement", os.path.join(project_root, "figures"))
plt.show()


### Plot: cumulative evidence reliability, with overall human accuracy as reference


In [ ]:
_set_plot_style(font_size=16)

fig, ax = plt.subplots(figsize=(8, 5.5))

# Cumulative evidence: the yellow/blue majority using only the first N draws
# of the card sequence -- a property of the cards alone, unrelated to what
# (if anything) any subject decided.
x_rel = draw_n_summary.index.to_numpy()
mean_rel = draw_n_summary["mean_across_subjects"].to_numpy()
sd_rel = draw_n_summary["sd_across_subjects"].to_numpy()
ax.fill_between(x_rel, mean_rel - sd_rel, mean_rel + sd_rel, color="steelblue", alpha=0.15, linewidth=0)
ax.plot(x_rel, mean_rel, color="steelblue", linewidth=2, marker="o", markersize=8, zorder=3,
        label="Cumulative evidence majority vs. final majority")

# Actual human accuracy: a single number averaged over ALL games (not a
# function of draw count -- accuracy doesn't vary by how many draws a game
# took), shown as a horizontal band (mean +/- SD across subjects).
acc_mean = per_subject_accuracy.mean()
acc_sd = per_subject_accuracy.std()
ax.axhspan(acc_mean - acc_sd, acc_mean + acc_sd, color="darkorange", alpha=0.15, linewidth=0, zorder=1)
ax.axhline(acc_mean, color="darkorange", linewidth=2, zorder=2,
           label=f"Actual human accuracy ({acc_mean:.0%} \u00b1 {acc_sd:.0%})")

ax.axhline(0.5, color="0.75", linestyle=":", linewidth=1, zorder=1, label="Chance (50%)")

ax.set_xlabel("Draw number")
ax.set_ylabel("Agreement with final majority / accuracy")
ax.set_title("Cumulative evidence reliability vs. overall human accuracy")
ax.set_xticks(range(1, 11))
ax.set_ylim(0.4, 1.0)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=True, loc="lower right", fontsize=12)
fig.tight_layout()

_save_figure(fig, "draw_n_reliability_vs_actual_accuracy", os.path.join(project_root, "figures"))
plt.show()

## Plot any subject's behaviour

`plot_multi_subjects_panel` draws a panel per subject, overlaying the short
and long horizon draw distributions, optionally with the model ensemble on
top. It produced Fig 8 in the paper and lives in `src/utils/plotting.py`, so
it can be called from anywhere.

Change `SUBJECTS` to any four subject IDs. Pass `show_simulation=False` if you
have not run the ensemble step and only want the human data.


In [ ]:
import os, sys, pickle
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from src.config.loader import load_config
from src.utils.plotting import plot_multi_subjects_panel

SUBJECTS = [17, 28, 43, 44]      # any four subject IDs
SHORT_TASK = "SB-XT-RPh----"     # the winning short horizon model
LONG_TASK = "LBE-T-RPhCL--"      # the winning long horizon model

root = os.path.abspath("..")


def load_ensemble(task, horizon):
    """Ensemble distributions and per subject metrics for one fitted model."""
    cfg = load_config(os.path.join(
        root, "data/simulation_configs", f"simulation_params_{task}.py"))
    with open(os.path.join(cfg.DATA_PATH, "ensemble_distribution_data.pkl"), "rb") as fh:
        ensemble = pickle.load(fh)
    metrics = pd.read_csv(os.path.join(cfg.DATA_PATH, "ensemble_metrics_summary.csv"))
    return ensemble, metrics


ens_short, met_short = load_ensemble(SHORT_TASK, "short")
ens_long, met_long = load_ensemble(LONG_TASK, "long")

plot_multi_subjects_panel(
    SUBJECTS,
    ens_short, ens_long,
    met_short, met_long,
    show_simulation=True,
    panel_labels=["a", "b", "c", "d"],
    path=os.path.join(root, "figures", "Human_Data"),
    filename="four_subjects_panel",
)
